# SI4006 · Entrega M3 — RAG agéntico sobre el sistema de riesgo crediticio

Este notebook implementa un sistema para consultar la documentación del proyecto. Recupera fragmentos relacionados con cada pregunta y usa una calculadora cuando hace falta operar con cifras. Si la respuesta no está en el corpus, debe abstenerse.

| Componente | Implementación |
| :-- | :-- |
| Corpus | Documentación propia de M1 y M2 |
| Comparación | Línea base sin recuperación y RAG ingenuo |
| RAG avanzado | BM25 + búsqueda densa, fusión RRF y reranking |
| Herramienta | Calculadora, activada por un router |
| Evaluación | Harness de M2, recuperación determinista y RAGAS |
| Juez | `Qwen2.5-7B-Instruct` en 4 bits |

Se ejecuta en Colab con GPU T4 y no requiere claves de pago. La primera ejecución descarga cerca de 15 GB de pesos del juez.

```
documentos ─► chunks 500/80 ─► embeddings MiniLM ─► Chroma ─┐
                                   BM25 sobre los chunks ───┤
                                                            ▼
pregunta ─► [densa top-10] + [BM25 top-10] ─► RRF ─► cross-encoder ─► top-4   ← técnicas avanzadas
                                                            ▼
                              router: ¿la pregunta pide una operación?        ← capa agéntica
                                 │ sí                              │ no
                 Qwen propone la expresión ─► calculadora          │
                                 │ resultado como pasaje           │
                                 ▼                                 ▼
                prompt base (con válvula) + contexto ─► Qwen2.5-1.5B ─► respuesta
```

## 0 · Setup

In [1]:
%pip install -q sentence-transformers chromadb rank_bm25 bitsandbytes accelerate "opentelemetry-api==1.42.1" "opentelemetry-sdk==1.42.1"
print('Listo.')

Listo.


In [2]:
import os, re, json, time, csv, random, inspect, gc
from pathlib import Path
import numpy as np, pandas as pd
import torch, transformers

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 70)

INTEGRANTES = ['Jaider España', 'Camila Martinez']
REPO_URL    = 'https://github.com/CamsMars/CREDIT'

RUTA_SALIDA = Path('m3_salida'); RUTA_SALIDA.mkdir(exist_ok=True)

EMB_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'   # el de M2
GEN_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'                                    # el de M2
RERANK_MODEL = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'

# Juez distinto del generador: evita la auto-preferencia y tiene capacidad para seguir la rúbrica.
# Si el 4 bits no carga (bitsandbytes), se intenta el respaldo en fp16; si tampoco, se usa el
# generador y el resumen final lo declara.
JUEZ_MODEL    = 'Qwen/Qwen2.5-7B-Instruct'
JUEZ_4BIT     = True
JUEZ_RESPALDO = 'Qwen/Qwen2.5-3B-Instruct'

print(f'{device} | torch {torch.__version__} | transformers {transformers.__version__}')

cuda | torch 2.11.0+cu130 | transformers 5.20.2


## 1 · Corpus

El corpus reúne ocho documentos técnicos elaborados por el equipo a partir de M1 y M2: diccionario de datos, limpieza, etiqueta, variables excluidas, modelo y umbral, reproducibilidad, harness y auditoría de sesgos. Es documentación propia de uso académico y del equipo.

Cada documento conserva `fuente` y `fecha`. El tamaño del corpus y el número de chunks se calculan en las celdas siguientes. La versión completa queda en `m3_salida/corpus_m3.json`.

In [3]:
corpus = [
 {'id':'doc1','fuente':'Diccionario de datos · perfiles.csv (2026)','fecha':'2026-08',
  'texto':(
   'El dataset perfiles.csv contiene 252.523 filas y 72 columnas con información de solicitantes de '
   'crédito. Las variables de identidad y vinculación son id_usuario, fecha_vinculacion, '
   'ultima_actualizacion, origen_registro e id_asesor. Las sociodemográficas son edad, sexo, ciudad, '
   'region, estrato, nivel_educativo, estado_civil, dependientes, propiedad_vivienda y tipo_vivienda. '
   'Las de ocupación e ingreso son profesion, codigo_ciiu, descripcion_ocupacion, tipo_empleo, '
   'ingreso_reportado, unidad_ingreso, volatilidad_ingreso y patrimonio_declarado. '
   'Las crediticias son deuda_total_actual, ratio_deuda_ingreso, cuota_mensual_actual, score_credito, '
   'score_bureau_externo, antiguedad_financiera, num_creditos_activos, num_entidades_reportadas, '
   'utilizacion_tdc y saldos_ultimos_6m. El comportamiento de pago está en moras_ultimos_24m, '
   'max_dias_mora_historico, meses_desde_ultima_mora, castigos_previos y consultas_centrales_6m. '
   'La solicitud está en monto_solicitado y plazo_solicitado. El desenlace está en meses_observados, '
   'tiempo_evento y evento. Las cinco columnas ruido_1 a ruido_5 son aleatorias puras y están en el '
   'archivo para distinguir a quien selecciona variables con criterio de quien lanza todas al modelo.')},

 {'id':'doc2','fuente':'Limpieza de datos · M1 secciones 3 y 4 (2026)','fecha':'2026-08',
  'texto':(
   'El archivo trae valores centinela: un número imposible que en realidad significa sin dato. '
   'score_credito usa -999, max_dias_mora_historico usa 999, num_entidades_reportadas usa 99, '
   'y dependientes y sesiones_app_mes usan -1. Todos se convierten a nulo antes de modelar. '
   'La columna fecha_vinculacion mezcla tres plantillas, ISO aaaa-mm-dd, dd/mm/aaaa y dd-Mon-aaaa, '
   'más dos fechas centinela, 1900-01-01 y 2031-12-31. Pasarlas juntas a un parser automático produce '
   'fechas silenciosamente incorrectas. La columna ciudad aparece escrita de 95 formas distintas para '
   '22 ciudades reales, con tildes, mayúsculas y sufijos como D.C. El ingreso viene en pesos o en miles '
   'de pesos: cuando falta unidad_ingreso y el valor es menor a 50.000 se asume que venía en miles, y '
   'la fila queda marcada en ingreso_unidad_imputada para poder auditarla. El patrimonio viene en USD '
   'en el 4 por ciento de las filas y se convierte con una TRM supuesta de 4.000 pesos por dólar. '
   'El score de buró externo ausente no se imputa: falta más cuando el cliente es peor, es decir que '
   'es MNAR, y rellenarlo con la mediana borraría justamente la señal.')},

 {'id':'doc3','fuente':'Definición de la etiqueta · M1 sección 5 (2026)','fecha':'2026-08',
  'texto':(
   'La variable evento igual a cero no significa buen cliente. Puede significar que el cliente pagó bien '
   'durante 48 meses o que solo se le observó cuatro. Mezclar ambos casos le enseña al modelo que un '
   'perfil riesgoso con poco seguimiento es seguro. La corrección es fijar un horizonte común de 12 meses: '
   'se conservan los clientes observados al menos 12 meses y se pregunta si incumplieron dentro de esos '
   '12 meses. Los clientes con menos seguimiento se descartan, porque de ellos no se conoce el desenlace '
   'y descartarlos es más honesto que llamarlos buenos. La tasa de incumplimiento a 12 meses resultante '
   'es de 3,65 por ciento, y esa prevalencia es el piso real de la métrica PR-AUC.')},

 {'id':'doc4','fuente':'Variables excluidas · M1 sección 6 (2026)','fecha':'2026-08',
  'texto':(
   'De las 72 columnas, 27 no entran al modelo y cada exclusión tiene un motivo. La variable sexo se '
   'excluye por ser una variable protegida. Las variables castigos_previos, clv_estimado y '
   'propension_churn se excluyen por fuga de información: se calculan después del desenlace. '
   'La variable score_politica_anterior codifica la decisión de la política vieja e introduce sesgo de '
   'selección. La variable segmento_mercado es la partición del simulador, no un hallazgo propio. '
   'La variable version_politica es constante. La variable id_asesor tiene 480 niveles y es un proxy '
   'operativo, no del cliente. Latitud y longitud traen 1,2 por ciento de coordenadas invertidas y 2 por '
   'ciento en el punto cero, y son redundantes con ciudad y región. La variable email_dominio es un proxy '
   'socioeconómico sin relación causal con el pago. Las variables de campaña, apertura_ofertas_pct y '
   'respuesta_campanas_previas, miden actividad comercial y no riesgo. Las cinco variables ruido se '
   'excluyen por ser aleatorias puras.')},

 {'id':'doc5','fuente':'Modelo y umbral de decisión · M1 parte B y M2 (2026)','fecha':'2026-10',
  'texto':(
   'El modelo es BETO, dccuchile/bert-base-spanish-wwm-cased, afinado con LoRA de rango 16 y alpha 32 '
   'sobre las proyecciones query y value de la atención, con dropout 0,1 y la cabeza de clasificación '
   'entrenada completa. Se entrena menos del 2 por ciento de los parámetros. La entrada no son columnas '
   'sino una ficha redactada en español de unos 1.200 caracteres por cliente. El entrenamiento submuestrea '
   'los negativos a tres por cada positivo, pero validación y test conservan la prevalencia real. '
   'La métrica principal es PR-AUC, porque con 3,65 por ciento de positivos la accuracy premia al modelo '
   'que no hace nada y el ROC-AUC se ve optimista. M1 reportó en test PR-AUC 0,1723, ROC-AUC 0,8278 y '
   'F1 0,2646, contra TF-IDF con regresión logística que alcanzó PR-AUC 0,1383, y contra BETO congelado '
   'con sonda sobre el token CLS que alcanzó 0,1221. El umbral de decisión se elige maximizando F1 sobre '
   'validación, nunca sobre test. M1 eligió 0,65. En M2 el umbral se volvió a derivar en 0,54 porque el '
   'pooler se había perdido y la escala de probabilidades quedó corrida.')},

 {'id':'doc6','fuente':'Incidente de reproducibilidad · M2 (2026)','fecha':'2026-10',
  'texto':(
   'El checkpoint de BETO viene de masked language modeling y no incluye la capa pooler, de modo que '
   'transformers la inicializa al azar en cada carga. LoRA no la entrenó, porque solo tocó query, value y '
   'la cabeza, así que quedó congelada en ese valor aleatorio. El método save_pretrained de PEFT guarda '
   'únicamente los parámetros entrenables, por lo que el pooler nunca se escribió a disco. Al recargar el '
   'adaptador en M2 se sortea otro pooler y la cabeza de clasificación queda leyendo features distintas de '
   'aquellas con las que aprendió. El harness lo detectó porque la tasa de alerta cayó a cero con el umbral '
   'heredado. Se midió el efecto probando doce semillas: el ROC-AUC en validación varía entre 0,2398 y '
   '0,7894 con el mismo adaptador y las mismas fichas. Un valor por debajo de 0,5 significa que el modelo '
   'ordena a los solicitantes al revés. Se eligió la mejor semilla, se guardó el pooler junto al adaptador '
   'en pooler.safetensors y se re-derivó el umbral sobre validación.')},

 {'id':'doc7','fuente':'Harness de evaluación y rúbrica del juez · M2 (2026)','fecha':'2026-10',
  'texto':(
   'El harness de M2 mide el sistema en tres dimensiones sobre un eval set propio. La dimensión 1 es una '
   'métrica clásica automática, la similitud coseno por embeddings con el modelo multilingüe '
   'paraphrase-multilingual-MiniLM-L12-v2. La dimensión 2 es un LLM-as-a-judge con rúbrica anclada de 1 a 5 '
   'y versionada en el repositorio. La dimensión 3 es una medida propia de aciertos de dominio con criterio '
   'explícito por caso. La firma es harness(eval_set, sistema) y se puede re-ejecutar sobre cualquier '
   'sistema que cumpla el contrato de entrada y salida. El juez se controla por tres sesgos conocidos: '
   'posición, longitud y auto-preferencia. El sesgo de posición se mitiga evaluando en ambos órdenes y '
   'declarando empate si el veredicto cambia. El sesgo de longitud se midió replicando el ataque de lista '
   'repetitiva de Zheng y otros, que duplica cada elemento de la lista reformulado sin aportar información '
   'nueva. El eval set de M2 tiene 14 casos, 10 gold y 4 adversariales, y está congelado en eval_set.json '
   'para que las comparaciones entre módulos sean válidas.')},

 {'id':'doc8','fuente':'Auditoría de sesgos del sistema · M2 (2026)','fecha':'2026-10',
  'texto':(
   'La auditoría desagrega el error del modelo por grupo sobre 1.500 fichas de test. La tasa de falsa '
   'alarma, es decir la proporción de solicitantes que sí habrían pagado pero quedan marcados como '
   'riesgosos, es de 29,3 por ciento entre los informales y de 6,0 por ciento entre los formales, casi '
   'cinco veces más. Por estrato va de 22,1 por ciento en estrato 1 a 7,4 por ciento en estrato 4. '
   'Un contrafactual sobre 300 fichas que cambia únicamente la palabra del estrato mueve la probabilidad '
   'de incumplimiento en menos de 0,001 en promedio y voltea el veredicto en el 0,3 por ciento de los '
   'casos, de modo que el estrato por sí solo no es el predictor: la penalización viene de las variables '
   'correlacionadas, sobre todo la informalidad y la volatilidad del ingreso, que el modelo sí usa. '
   'En términos estadísticos no es un error, porque la población informal incumple más, 4,0 por ciento '
   'contra 2,9 por ciento. En términos de lo que le ocurre a una persona, significa que a un informal que '
   'habría pagado se le niega el crédito cinco veces más a menudo que a un formal en su misma situación.')},
]

print(f'{len(corpus)} documentos | {sum(len(d["texto"]) for d in corpus):,} caracteres')

8 documentos | 8,172 caracteres


## 2 · Chunking e índice

Usamos chunks de **500 caracteres con 80 de solapamiento**. Con 400, algunas definiciones quedaban partidas; 500 conserva explicaciones de varias frases sin añadir demasiado contexto repetido.

Para los embeddings usamos `paraphrase-multilingual-MiniLM-L12-v2`, el mismo modelo de la dimensión 1 del harness de M2. Así mantenemos el criterio de similitud y la búsqueda en el mismo espacio. Además, es liviano y funciona con el corpus en español.

Dejamos **top-k = 4**. En las pruebas, k=3 omitía información necesaria para algunas preguntas que cruzan documentos y k=5 añadía fragmentos que el generador tendía a repetir.

In [4]:
CHUNK_SIZE, OVERLAP, TOP_K = 500, 80, 4

def partir(texto, size=CHUNK_SIZE, overlap=OVERLAP):
    out, i = [], 0
    while i < len(texto):
        out.append(texto[i:i+size]); i += size - overlap
    return out

chunks, metadatos, ids = [], [], []
for d in corpus:
    for j, ch in enumerate(partir(d['texto'])):
        chunks.append(ch)
        metadatos.append({'fuente': d['fuente'], 'fecha': d['fecha'], 'doc_id': d['id']})
        ids.append(f"{d['id']}_c{j}")

print(f'{len(corpus)} documentos -> {len(chunks)} chunks')

8 documentos -> 23 chunks


In [5]:
from sentence_transformers import SentenceTransformer
import chromadb

st = SentenceTransformer(EMB_MODEL, device=device)
cliente = chromadb.Client()
NOMBRE_COL = 'm3_corpus'
try: cliente.delete_collection(NOMBRE_COL)
except Exception: pass
col = cliente.create_collection(NOMBRE_COL, metadata={'hnsw:space': 'cosine'})

E = st.encode(chunks, show_progress_bar=False)
col.add(ids=ids, documents=chunks, metadatas=metadatos, embeddings=E.tolist())
print('Indexados', col.count(), 'chunks')


def buscar_densa(consulta, k=TOP_K):
    r = col.query(query_embeddings=st.encode([consulta]).tolist(), n_results=k)
    return [{'id': i, 'texto': doc, 'fuente': m['fuente'], 'score': round(1 - d, 3)}
            for d, i, doc, m in zip(r['distances'][0], r['ids'][0], r['documents'][0], r['metadatas'][0])]

for r in buscar_densa('¿qué umbral usa el modelo?', 2):
    print(f"[{r['score']}] {r['id']}: {r['texto'][:80]}...")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexados 23 chunks
[0.585] doc7_c0: El harness de M2 mide el sistema en tres dimensiones sobre un eval set propio. L...
[0.511] doc5_c0: El modelo es BETO, dccuchile/bert-base-spanish-wwm-cased, afinado con LoRA de ra...


## 3 · Generador, juez, válvula de escape y RAG ingenuo

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer


def _cargar_causal(nombre, **kw):
    dt = torch.float16 if device == 'cuda' else torch.float32
    try:
        return AutoModelForCausalLM.from_pretrained(nombre, dtype=dt, **kw)
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(nombre, torch_dtype=dt, **kw)


gen_tok = AutoTokenizer.from_pretrained(GEN_MODEL)
gen_model = _cargar_causal(GEN_MODEL).to(device).eval()


@torch.no_grad()
def _generar_chat(tok, model, msgs, tools, max_new_tokens):
    """Genera a partir de una conversación. Con `tools`, la plantilla de chat de Qwen anuncia las
    herramientas en su formato nativo de function calling."""
    p = tok.apply_chat_template(msgs, tools=tools, tokenize=False, add_generation_prompt=True)
    ent = tok(p, return_tensors='pt').to(model.device)
    out = model.generate(**ent, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0][ent['input_ids'].shape[1]:], skip_special_tokens=True).strip()


def _generar(tok, model, system, user, max_new_tokens):
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]
    return _generar_chat(tok, model, msgs, None, max_new_tokens)


def generar(system, user, max_new_tokens=180):
    """El generador de los cuatro sistemas."""
    return _generar(gen_tok, gen_model, system, user, max_new_tokens)


def generar_chat(msgs, tools=None, max_new_tokens=180):
    """El mismo generador, con conversación de varios turnos (lo usa el agente)."""
    return _generar_chat(gen_tok, gen_model, msgs, tools, max_new_tokens)


def _cargar_juez():
    if device != 'cuda':
        print('Sin GPU: se usa el generador como juez.')
        return gen_tok, gen_model, f'{GEN_MODEL} (el generador)'
    intentos = ([(JUEZ_MODEL, '4bit')] if JUEZ_4BIT else [(JUEZ_MODEL, 'fp16')]) + [(JUEZ_RESPALDO, 'fp16')]
    for nombre, modo in intentos:
        try:
            tok = AutoTokenizer.from_pretrained(nombre)
            if modo == '4bit':
                from transformers import BitsAndBytesConfig
                q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                       bnb_4bit_compute_dtype=torch.float16)
                m = AutoModelForCausalLM.from_pretrained(nombre, quantization_config=q, device_map={'': 0})
            else:
                m = _cargar_causal(nombre).to(device)
            return tok, m.eval(), f'{nombre} ({modo})'
        except Exception as e:
            print(f'No se pudo cargar {nombre} en {modo}: {type(e).__name__}: {str(e)[:150]}')
            gc.collect()
            torch.cuda.empty_cache()
    print('Se usa el generador como juez: hay auto-preferencia y el juez es débil.')
    return gen_tok, gen_model, f'{GEN_MODEL} (el generador)'


juez_tok, juez_model, JUEZ_DESCRIPCION = _cargar_juez()
JUEZ_ES_GENERADOR = juez_model is gen_model


def generar_juez(system, user, max_new_tokens=3):
    """El juez del harness y de RAGAS."""
    return _generar(juez_tok, juez_model, system, user, max_new_tokens)


print('Generador:', GEN_MODEL)
print('Juez     :', JUEZ_DESCRIPCION)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

... (10 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Generador: Qwen/Qwen2.5-1.5B-Instruct
Juez     : Qwen/Qwen2.5-7B-Instruct (4bit)


In [7]:
VALVULA = 'No tengo esa información en mis fuentes.'

SYSTEM_RAG = (
    'Eres un asistente que responde SOLO con base en el contexto proporcionado. '
    f'Si la respuesta no está en el contexto, responde exactamente: "{VALVULA}" '
    'No inventes datos. Menciona la fuente que usaste. Sé breve.')


def _contexto(pasajes):
    return '\n\n'.join(f"[Fuente: {p['fuente']}]\n{p['texto']}" for p in pasajes)


def sistema_sin_rag(pregunta):
    """Línea base: el generador respondiendo de memoria, sin recuperación."""
    return generar('Responde de forma breve, correcta y clara.', pregunta)


def sistema_rag(pregunta, k=TOP_K, devolver_contexto=False):
    pasajes = buscar_densa(pregunta, k)
    r = generar(SYSTEM_RAG, f'Contexto:\n{_contexto(pasajes)}\n\nPregunta: {pregunta}')
    return (r, pasajes) if devolver_contexto else r


q = '¿Qué umbral de decisión eligió M1 y por qué cambió en M2?'
print('SIN RAG :', sistema_sin_rag(q)[:200])
print('\nCON RAG :', sistema_rag(q)[:300])

SIN RAG : M1 eligió un umbral de 0.5 para tomar decisiones. Cambió a un umbral más alto (por ejemplo, 0.7) en M2 debido a una mayor confianza en la precisión del modelo.

CON RAG : M1 eligió un umbral de decisión de 0,65. En M2, este umbral fue cambiado a 0,54 debido a que el pooler se había perdido y la escala de probabilidades quedó corrida.


## 4 · Diagnóstico: dónde falla el RAG ingenuo

Antes de elegir técnicas, revisamos si el chunk que contiene cada respuesta aparecía entre los primeros resultados de la búsqueda densa. Consideramos falla confirmada si quedaba fuera del top-4 y falla débil si llegaba, pero no en primer lugar.

El diagnóstico no confirmó todas las hipótesis iniciales: `-999` y `PR-AUC` ya aparecían en primera posición. La consulta coloquial sobre el incidente del `pooler`, sí tuvo problemas, el fragmento relevante no quedó entre los tres primeros resultados.

In [8]:
CONSULTAS_DIAGNOSTICO = [
 {'consulta': '¿qué significa el centinela -999?', 'chunk_esperado': 'doc2_c0',
  'hipotesis': 'término exacto',
  'nota': 'el literal -999 podría no pesar en el espacio semántico'},
 {'consulta': '¿cuánto vale PR-AUC del modelo afinado?', 'chunk_esperado': 'doc5_c1',
  'hipotesis': 'término exacto',
  'nota': 'la sigla PR-AUC aparece en doc3 y doc5; solo doc5_c1 trae la cifra'},
 {'consulta': 'mi modelo da resultados distintos cada vez que lo cargo, ¿por qué?', 'chunk_esperado': 'doc6_c0',
  'hipotesis': 'consulta coloquial',
  'nota': 'ninguna palabra coincide con el doc6; depende de que el embedding capte el significado'},
 {'consulta': '¿por qué no se usa el sexo del solicitante?', 'chunk_esperado': 'doc4_c0',
  'hipotesis': 'top-k con ruido',
  'nota': 'varios chunks hablan de variables; el motivo podría no quedar arriba'},
 {'consulta': '¿a quién perjudica más el modelo?', 'chunk_esperado': 'doc8_c0',
  'hipotesis': 'consulta abstracta',
  'nota': 'requiere el chunk con las tasas por grupo de la auditoría'},
]


def rango_en(pasajes, chunk_id):
    ids_ = [p['id'] for p in pasajes]
    return ids_.index(chunk_id) + 1 if chunk_id in ids_ else None


def estado_falla(r):
    if r is None or r > TOP_K:
        return 'falla confirmada'
    return 'falla débil' if r > 1 else 'no falla'


for c in CONSULTAS_DIAGNOSTICO:
    r = rango_en(buscar_densa(c['consulta'], 10), c['chunk_esperado'])
    c['rango_densa'] = r if r is not None else '>10'
    c['estado'] = estado_falla(r)

diag = pd.DataFrame(CONSULTAS_DIAGNOSTICO)
display(diag[['consulta', 'hipotesis', 'chunk_esperado', 'rango_densa', 'estado']])

,consulta,hipotesis,chunk_esperado,rango_densa,estado
0,¿qué significa el centinela -999?,término exacto,doc2_c0,1,no falla
1,¿cuánto vale PR-AUC del modelo afinado?,término exacto,doc5_c1,1,no falla
2,"mi modelo da resultados distintos cada vez que lo cargo, ¿por qué?",consulta coloquial,doc6_c0,>10,falla confirmada
3,¿por qué no se usa el sexo del solicitante?,top-k con ruido,doc4_c0,1,no falla
4,¿a quién perjudica más el modelo?,consulta abstracta,doc8_c0,4,falla débil


**Técnica 1 · búsqueda híbrida BM25 + densa con RRF.** Ataca las consultas cuyo vocabulario no se parece al del chunk en el espacio denso, y protege los literales (`-999`, `PR-AUC`, `pooler`) que
BM25 sí puntúa. RRF fusiona las dos listas por rango, sin calibrar escalas distintas. Que la hipótesis de término exacto no se haya confirmado con la búsqueda densa es un dato: si la híbrida no mejora esas consultas, la lectura honesta lo dice.

**Técnica 2 · reranking con cross-encoder.** Ataca las fallas débiles, donde el chunk correcto está entre los candidatos pero no arriba. El bi-encoder embebe consulta y chunk por separado; el cross-encoder los lee juntos y reordena. Se recuperan 10 candidatos y se devuelven los 4 mejores.
Después del reranker, una tabla compara la posición del chunk correcto en los cuatro recuperadores.

In [9]:
from rank_bm25 import BM25Okapi

def tokenizar(t):
    return re.findall(r'[\w\-]+', t.lower())

bm25 = BM25Okapi([tokenizar(c) for c in chunks])

def buscar_bm25(consulta, k=TOP_K):
    s = bm25.get_scores(tokenizar(consulta))
    top = np.argsort(-s)[:k]
    return [{'id': ids[i], 'texto': chunks[i], 'fuente': metadatos[i]['fuente'],
             'score': round(float(s[i]), 3)} for i in top]


def rrf(listas, k_rrf=60, k=TOP_K):
    """Reciprocal Rank Fusion: cada lista vota por rango, no por score."""
    puntos, ref = {}, {}
    for lista in listas:
        for rango, p in enumerate(lista):
            puntos[p['id']] = puntos.get(p['id'], 0) + 1 / (k_rrf + rango + 1)
            ref[p['id']] = p
    orden = sorted(puntos, key=puntos.get, reverse=True)[:k]
    return [{**ref[i], 'score': round(puntos[i], 4)} for i in orden]


def buscar_hibrida(consulta, k=TOP_K, n=10):
    return rrf([buscar_densa(consulta, n), buscar_bm25(consulta, n)], k=k)


c = '¿qué significa el centinela -999?'
print('densa  :', [p['id'] for p in buscar_densa(c, 4)])
print('bm25   :', [p['id'] for p in buscar_bm25(c, 4)])
print('híbrida:', [p['id'] for p in buscar_hibrida(c, 4)])

densa  : ['doc2_c0', 'doc2_c2', 'doc2_c1', 'doc6_c1']
bm25   : ['doc2_c0', 'doc6_c1', 'doc8_c2', 'doc3_c0']
híbrida: ['doc2_c0', 'doc6_c1', 'doc2_c2', 'doc7_c0']


In [10]:
from sentence_transformers import CrossEncoder

USAR_RERANKER = True
reranker = CrossEncoder(RERANK_MODEL, device=device) if USAR_RERANKER else None


def rerankear(consulta, pasajes, k=TOP_K):
    if reranker is None or not pasajes:
        return pasajes[:k]
    s = reranker.predict([(consulta, p['texto']) for p in pasajes])
    orden = np.argsort(-np.asarray(s))[:k]
    return [{**pasajes[i], 'score': round(float(s[i]), 3)} for i in orden]


def recuperar_avanzado(consulta, k=TOP_K, n=10):
    return rerankear(consulta, buscar_hibrida(consulta, k=n, n=n), k=k)


print('avanzado:', [p['id'] for p in recuperar_avanzado(c, 4)])

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

avanzado: ['doc2_c0', 'doc7_c1', 'doc2_c2', 'doc2_c1']


In [ ]:
RECUPERADORES = {
    'densa':    lambda q: buscar_densa(q, 10),
    'bm25':     lambda q: buscar_bm25(q, 10),
    'híbrida':  lambda q: buscar_hibrida(q, k=10, n=10),
    'avanzado': lambda q: recuperar_avanzado(q, k=10, n=10),
}
filas = []
for c in CONSULTAS_DIAGNOSTICO:
    fila = {'consulta': c['consulta'][:48], 'hipotesis': c['hipotesis'], 'chunk_esperado': c['chunk_esperado']}
    for nombre, fn in RECUPERADORES.items():
        r = rango_en(fn(c['consulta']), c['chunk_esperado'])
        fila[nombre] = r if r is not None else '>10'
    filas.append(fila)
rangos_diag = pd.DataFrame(filas)
display(rangos_diag)
print(f'Una posición mayor que {TOP_K} significa que el chunk no llega al generador.')

,consulta,hipotesis,chunk_esperado,densa,bm25,híbrida,avanzado
0,¿qué significa el centinela -999?,término exacto,doc2_c0,1,1,1,1
1,¿cuánto vale PR-AUC del modelo afinado?,término exacto,doc5_c1,1,2,1,1
2,mi modelo da resultados distintos cada vez que l,consulta coloquial,doc6_c0,>10,1,8,1
3,¿por qué no se usa el sexo del solicitante?,top-k con ruido,doc4_c0,1,1,1,1
4,¿a quién perjudica más el modelo?,consulta abstracta,doc8_c0,4,7,3,1


Una posición mayor que 4 significa que el chunk no llega al generador.


## 5 · `sistema_rag_avanzado`

In [12]:
def sistema_rag_avanzado(pregunta, k=TOP_K, devolver_contexto=False):
    pasajes = recuperar_avanzado(pregunta, k)
    r = generar(SYSTEM_RAG, f'Contexto:\n{_contexto(pasajes)}\n\nPregunta: {pregunta}')
    return (r, pasajes) if devolver_contexto else r


for c in CONSULTAS_DIAGNOSTICO[:2]:
    print(f"\n{c['consulta']}")
    print('  ingenuo :', sistema_rag(c['consulta'])[:150])
    print('  avanzado:', sistema_rag_avanzado(c['consulta'])[:150])


¿qué significa el centinela -999?
  ingenuo : El centinela -999 en el contexto proporcionado se refiere a un valor imposible que representa el estado de sin dato o ausencia de información en la co
  avanzado: El centinela -999 en el contexto proporcionado se refiere a un valor imposible que representa sin dato en los archivos utilizados. Es común en sistema

¿cuánto vale PR-AUC del modelo afinado?
  ingenuo : PR-AUC del modelo afinado es de 0,1723.
  avanzado: PR-AUC del modelo afinado es de 0,1723.


## 6 · Capa agéntica

El sistema tiene dos herramientas disponibles durante los experimentos: una calculadora y el retriever avanzado. Probamos primero un agente libre, en el que el generador decidía cuándo llamar herramientas. Ninguno de los tres formatos evaluados produjo llamadas confiables, a veces omitía cálculos, hacía búsquedas innecesarias o se abstenía aunque el RAG avanzado podía responder. En la evaluación bajó de 9/10 a 6/10 casos gold y aumentó la latencia. Conservamos esa versión como `sistema_agente_libre` para la ablación.

La versión final usa un **router basado en reglas** para detectar preguntas que requieren operaciones. El generador propone la expresión, la calculadora la ejecuta y el resultado se agrega al contexto con fuente `calculadora`. La respuesta final mantiene el prompt del RAG avanzado.

El retriever-herramienta no quedó en el sistema final, la recuperación avanzada ya encontró el documento correcto en 13 de 13 casos y la herramienta solo añadía latencia en el agente libre.

El router se diseñó con conocimiento del eval set, por lo que su resultado no demuestra que generalice. El parseo admite formato nativo, JSON o expresión en texto; hay límite de pasos, registro de llamadas y manejo de errores de cálculo.

In [13]:
def calculadora(expresion):
    if not re.fullmatch(r'[\d\s\+\-\*/\(\)\.,%]+', str(expresion) or ''):
        return 'error: solo se admiten expresiones aritméticas'
    try:
        return str(round(eval(str(expresion).replace(',', '.').replace('%', '/100'),
                              {'__builtins__': {}}, {}), 4))
    except Exception as e:
        return f'error: {e}'


_PASAJES_TOOL = []   # pasajes que trae buscar_en_corpus, para que cuenten como contexto


def buscar_en_corpus(consulta):
    ps = recuperar_avanzado(str(consulta), 3)
    _PASAJES_TOOL.extend(ps)
    return '\n'.join(f"[{p['fuente']}] {p['texto']}" for p in ps)


TOOLS = {'calculadora': calculadora, 'buscar_en_corpus': buscar_en_corpus}

# Esquemas en el formato de function calling que entiende la plantilla de chat de Qwen.
ESQUEMAS = [
 {'type': 'function', 'function': {
   'name': 'calculadora',
   'description': ('Evalúa una expresión aritmética y devuelve el resultado. Úsala SIEMPRE que la '
                   'pregunta pida una operación (cuántas veces mayor, diferencia, resta, suma, cuántos '
                   'faltan) cuyo resultado no esté escrito en el contexto. No la uses si el número ya '
                   'aparece en el contexto.'),
   'parameters': {'type': 'object',
                  'properties': {'expresion': {'type': 'string',
                                               'description': 'Expresión con punto decimal, p. ej. "18.0 / 4.5"'}},
                  'required': ['expresion']}}},
 {'type': 'function', 'function': {
   'name': 'buscar_en_corpus',
   'description': ('Busca más pasajes en la documentación del proyecto. Úsala solo si al contexto le '
                   'falta un dato que la pregunta pide.'),
   'parameters': {'type': 'object',
                  'properties': {'consulta': {'type': 'string', 'description': 'Qué buscar'}},
                  'required': ['consulta']}}},
]


def ejecutar_tool(nombre, args):
    fn = TOOLS.get(nombre)
    if fn is None:
        return f'error: no existe la herramienta {nombre}'
    try:
        return str(fn(**args))[:600]
    except TypeError:
        return f'error: argumentos inválidos para {nombre}'


def extraer_json(t):
    i, j = t.find('{'), t.rfind('}')
    if i == -1 or j == -1 or j < i:
        return None
    try:
        return json.loads(t[i:j+1])
    except Exception:
        return None


def extraer_llamada(t):
    """Devuelve (nombre, args) si la salida pide una herramienta, o None. Tolerante: acepta el
    formato nativo <tool_call>{"name", "arguments"}</tool_call> y el JSON suelto {"tool", "args"}."""
    obj = None
    m = re.search(r'<tool_call>\s*(\{.*?\})\s*(?:</tool_call>|$)', t, re.S)
    if m:
        try:
            obj = json.loads(m.group(1))
        except Exception:
            obj = None
    if obj is None:
        obj = extraer_json(t)
    if not isinstance(obj, dict):
        return None
    nombre = obj.get('name') or obj.get('tool')
    args = obj.get('arguments', obj.get('args', {})) or {}
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except Exception:
            args = {}
    if not isinstance(nombre, str) or nombre not in TOOLS:
        return None
    return nombre, (args if isinstance(args, dict) else {})


print('Herramientas:', list(TOOLS))
print('prueba calculadora:', calculadora('29.3 / 6.0'), '| inyección:', calculadora('__import__("os")'))

Herramientas: ['calculadora', 'buscar_en_corpus']
prueba calculadora: 4.8833 | inyección: error: solo se admiten expresiones aritméticas


In [ ]:
MAX_PASOS = 3
SYSTEM_AGENTE = SYSTEM_RAG     # el mismo prompt base del RAG


def _limpiar(texto):
    return re.sub(r'</?tool_call>', '', texto).strip()


def _sin_repetidos(pasajes):
    vistos, out = set(), []
    for p in pasajes:
        if p['id'] not in vistos:
            vistos.add(p['id']); out.append(p)
    return out


# ---------------------------------------------------------------------------------------------------------------------------------------
# Versión 1 · agente libre: el generador decide solo qué herramienta usar, evidencia de por qué el sistema final usa un router.
# ---------------------------------------------------------------------------------------------------------------------------------------
def sistema_agente_libre(pregunta, devolver_contexto=False, verbose=False):
    _PASAJES_TOOL.clear()
    pasajes = recuperar_avanzado(pregunta, TOP_K)
    msgs = [{'role': 'system', 'content': SYSTEM_AGENTE},
            {'role': 'user', 'content': f'Contexto:\n{_contexto(pasajes)}\n\nPregunta: {pregunta}'}]
    llamadas = []

    def _salida(texto):
        texto = _limpiar(texto)
        if not devolver_contexto:
            return texto
        return texto, _sin_repetidos(pasajes + _PASAJES_TOOL), llamadas

    for _ in range(MAX_PASOS):
        salida = generar_chat(msgs, tools=ESQUEMAS)
        pedido = extraer_llamada(salida)
        if pedido is None:
            return _salida(salida)
        nombre, args = pedido
        obs = ejecutar_tool(nombre, args)
        llamadas.append({'tool': nombre, 'args': args, 'obs': obs[:200]})
        if verbose:
            print(f'   -> {nombre}({args}) = {obs[:70]}')
        msgs.append({'role': 'assistant', 'content': '',
                     'tool_calls': [{'type': 'function', 'function': {'name': nombre, 'arguments': args}}]})
        msgs.append({'role': 'tool', 'name': nombre, 'content': obs})
    return _salida(generar_chat(msgs))


# ---------------------------------------------------------------------------------------------
# Versión final · router + calculadora.
# ---------------------------------------------------------------------------------------------
PATRONES_OPERACION = [
    r'\bcu[aá]nt[oa]s\s+veces\b',
    r'\bdiferencia\b',
    r'\bpuntos\s+porcentuales\b',
    r'\bcu[aá]nt[oa]s\b[^?]*\b(faltan|quedan|sobran|entran|restan|suman|más|menos)\b',
    r'\b(suma|resta|divide|multiplica|promedio)\b',
]


def necesita_calculo(pregunta):
    """Router: True si la pregunta pide una operación aritmética."""
    q = pregunta.lower()
    return any(re.search(p, q) for p in PATRONES_OPERACION)


ESQUEMA_CALC = [e for e in ESQUEMAS if e['function']['name'] == 'calculadora']
SYSTEM_EXPRESION = ('Traduces preguntas a operaciones aritméticas. Responde SOLO con la operación, '
                    'sin texto, sin resultado y sin unidades.')


def _expresion_valida(t):
    t = str(t).replace(',', '.')
    m = re.search(r'[\d\(][\d\.\s\+\-\*/\(\)]*[\d\)]', t)
    if not m:
        return None
    e = m.group(0).strip()
    return e if re.search(r'[\d\)]\s*[\+\-\*/]\s*[\d\(]', e) else None


def _proponer_calculo(pasajes, pregunta, nota=''):
    """El generador propone el argumento de la calculadora. Primero en formato nativo; si no emite
    la llamada, se le pide la expresión en texto. Devuelve (expresion, via) o (None, None)."""
    ctx = f'Contexto:\n{_contexto(pasajes)}\n\nPregunta: {pregunta}' + (f'\n\n{nota}' if nota else '')
    salida = generar_chat([{'role': 'system', 'content': SYSTEM_AGENTE},
                           {'role': 'user', 'content': ctx}], tools=ESQUEMA_CALC, max_new_tokens=60)
    pedido = extraer_llamada(salida)
    if pedido and pedido[0] == 'calculadora':
        e = _expresion_valida(pedido[1].get('expresion', ''))
        if e:
            return e, 'nativa'
    salida = generar(SYSTEM_EXPRESION,
                     ctx + '\n\nEscribe SOLO la operación con los números del contexto y punto decimal. '
                           'Ejemplos: 18.0 / 4.5 · 72 - 27', max_new_tokens=30)
    e = _expresion_valida(salida)
    return (e, 'texto') if e else (None, None)


def sistema_final(pregunta, devolver_contexto=False, verbose=False):
    """RAG avanzado + calculadora decidida por un router. Con devolver_contexto=True devuelve
    (respuesta, pasajes, llamadas). Sin operación, es idéntico a sistema_rag_avanzado."""
    pasajes = recuperar_avanzado(pregunta, TOP_K)
    llamadas, extra, nota = [], '', ''
    if necesita_calculo(pregunta):
        for _ in range(MAX_PASOS - 1):
            expr, via = _proponer_calculo(pasajes, pregunta, nota)
            if expr is None:
                break
            obs = ejecutar_tool('calculadora', {'expresion': expr})
            llamadas.append({'tool': 'calculadora', 'args': {'expresion': expr}, 'obs': obs, 'via': via})
            if verbose:
                print(f'   -> calculadora({expr!r}) = {obs}   [propuesta {via}]')
            if not obs.startswith('error'):
                extra = f'\n\n[Fuente: calculadora]\n{expr} = {obs}'
                break
            nota = f'La operación "{expr}" dio {obs}. Corrígela.'
    r = generar(SYSTEM_RAG, f'Contexto:\n{_contexto(pasajes)}{extra}\n\nPregunta: {pregunta}')
    return (r, pasajes, llamadas) if devolver_contexto else r


# Verificación temprana: La plantilla de chat acepta una conversación con llamada y resultado de herramienta. Si falla aquí, no falla a mitad de la tabla evolutiva.
_prueba = [{'role': 'system', 'content': SYSTEM_AGENTE}, {'role': 'user', 'content': '¿Cuánto es 2 + 2?'},
           {'role': 'assistant', 'content': '', 'tool_calls': [
               {'type': 'function', 'function': {'name': 'calculadora', 'arguments': {'expresion': '2 + 2'}}}]},
           {'role': 'tool', 'name': 'calculadora', 'content': '4'}]
_txt = gen_tok.apply_chat_template(_prueba, tools=ESQUEMAS, tokenize=False, add_generation_prompt=True)
assert '<tool_call>' in _txt and '<tool_response>' in _txt and 'calculadora' in _txt, \
    'La plantilla de chat del generador no reconoce el formato de herramientas'
print('Plantilla de herramientas: OK')

Plantilla de herramientas: OK


In [ ]:
DEMOS = [
 (True,  'Según la auditoría, ¿cuántas veces mayor es la falsa alarma de los informales frente a los formales?'),
 (True,  'Si a un cliente solo se le observó 4 meses, ¿cuántos meses le faltan para el horizonte de la etiqueta?'),
 (False, '¿Qué modelo base usa el sistema?'),
]
demos_agente = []
for necesita, q in DEMOS:
    print(f'\n[{"necesita herramienta" if necesita else "no la necesita"}] {q}')
    r, _, ll = sistema_final(q, devolver_contexto=True, verbose=True)
    llamo = len(ll) > 0
    veredicto = 'correcto' if llamo == necesita else ('NO la llamó' if necesita else 'la llamó DE MÁS')
    print(f'   herramientas usadas: {len(ll)} -> {veredicto}')
    print('  ', r[:220])
    demos_agente.append({'consulta': q, 'necesita': necesita, 'llamadas': [l['tool'] for l in ll],
                         'expresion': ll[-1]['args']['expresion'] if ll else '',
                         'veredicto': veredicto, 'respuesta': r})


[necesita herramienta] Según la auditoría, ¿cuántas veces mayor es la falsa alarma de los informales frente a los formales?
   -> calculadora('(29.3 * 1.5) / (6.0 * 1.5)') = 4.8833   [propuesta texto]
   herramientas usadas: 1 -> correcto
   Según la auditoría, la falsa alarma de los informales es aproximadamente cinco veces mayor que la de los formales. Esto se refleja en una tasa de falsa alarma de 29,3% entre los informales y 6,0% entre los formales.

[necesita herramienta] Si a un cliente solo se le observó 4 meses, ¿cuántos meses le faltan para el horizonte de la etiqueta?
   herramientas usadas: 0 -> NO la llamó
   Si a un cliente solo se le observó 4 meses, le faltarían 8 meses para completar el horizonte de la etiqueta de 12 meses.

[no la necesita] ¿Qué modelo base usa el sistema?
   herramientas usadas: 0 -> correcto
   El sistema utiliza el modelo BASE, dccuchile/bert-base-spanish-wwm-cased, afinado con LoRA de rango 16 y alpha 32 sobre las proyecciones query y value de la 

## 7 · El harness de M2

Mantenemos la estructura del harness de M2: tres dimensiones, `harness(eval_set, sistema)` y rúbrica de 1 a 5. Está incluida aquí para que M3 se ejecute de forma independiente.

La dimensión 2 se reporta dos veces. `D2 · juez M2` es la vara original sin tocar: el mismo juez de 1.5B, las mismas cuatro preguntas, la misma regla. Como esa vara falló su propia prueba de cordura, se agrega `D2 · juez v1.1` (Qwen 7B), que es la que usa D3. Las dos se aplican igual a las seis versiones, así que el delta entre filas sigue siendo limpio.

La v1.1 cambia tres cosas, todas en el instrumento. El juez ya no es el generador. Los casos de válvula se califican con una regla determinista, porque el juez les ponía 2 a abstenciones correctas. Y la pregunta de invención se invirtió y el juez ve también el contexto recuperado, para no castigar como invento un dato que el sistema tomó de las fuentes.

El criterio de dominio sí cambia respecto a M2, porque la tarea cambió: M1 clasificaba fichas y M3
responde preguntas sobre un corpus. La asignación lo contempla para tareas de clasificación.

In [ ]:
# RÚBRICA DEL JUEZ · M3 v1.1
RUBRICA = """Evalúas RESPUESTAS de un asistente que consulta la documentación de un sistema de
riesgo crediticio.

Niveles:
5 = correcta respecto a la referencia, sin datos inventados y completa.
4 = correcta y sin inventos, pero deja fuera parte de lo que se preguntó.
3 = parcialmente correcta: acierta en algo y se equivoca o divaga en el resto.
2 = incorrecta respecto a la referencia.
1 = inventa datos que no están en la referencia ni en el contexto.

El nivel se deriva de cuatro preguntas verificables, en este orden:
  1. ¿La respuesta afirma datos que no están ni en la referencia ni en el contexto?  sí -> 1
  2. ¿La respuesta coincide en lo esencial con la referencia?                        no -> 2
  3. ¿Responde sin divagar ni contradecirse?                                         no -> 3
  4. ¿Responde todo lo que se preguntó?                                              no -> 4
  Las cuatro favorables                                                                 -> 5

Válvula de escape (regla determinista, sin juez):
  - la referencia es la válvula y el sistema se abstiene                                -> 5
  - la referencia es la válvula y el sistema responde algo                              -> 1
  - la respuesta está en el corpus y el sistema se abstiene                             -> 2
La extensión no cuenta: una respuesta más larga no es mejor."""
# -------------------------------------------------------------------------------------------------

PREGUNTAS_JUEZ = [
 ('inventa',  '¿La RESPUESTA A EVALUAR afirma algún dato, cifra, nombre, fecha o entidad que NO aparece '
              'ni en la RESPUESTA DE REFERENCIA ni en el CONTEXTO RECUPERADO?'),
 ('correcta', '¿La RESPUESTA A EVALUAR dice en lo esencial lo mismo que la RESPUESTA DE REFERENCIA?'),
 ('centrada', '¿La RESPUESTA A EVALUAR responde la PREGUNTA DEL USUARIO sin divagar ni contradecirse?'),
 ('completa', '¿La RESPUESTA A EVALUAR incluye todos los datos de la RESPUESTA DE REFERENCIA?'),
]

CONTADOR_FALLBACK = {'n': 0, 'total': 0, 'deterministas': 0}
DEBUG_JUEZ = False


def es_valvula(respuesta):
    t = str(respuesta).lower()
    return 'no tengo esa información' in t or 'no tengo esa informacion' in t


def _si_no(texto):
    """Acepta sí/no en español e inglés; None si no aparece ninguna de las dos."""
    m = re.search(r'\b(S[IÍ]|YES|CORRECTO|VERDADERO|NO|FALSE|INCORRECTO)\b', texto.strip().upper())
    return None if not m else m.group(1) in ('SI', 'SÍ', 'YES', 'CORRECTO', 'VERDADERO')


def juez_puntua(pregunta, respuesta, referencia, contexto=None):
    """Rúbrica multi-criterio: el nivel 1-5 sale de preguntas binarias. Los casos de válvula
    se resuelven con una regla determinista, sin llamar al juez."""
    if referencia == VALVULA or es_valvula(respuesta):
        CONTADOR_FALLBACK['deterministas'] += 1
        if referencia == VALVULA:
            p, razon = (5, 'se abstuvo') if es_valvula(respuesta) else (1, 'respondió algo que no está en las fuentes')
        else:
            p, razon = 2, 'se abstuvo aunque la respuesta está en el corpus'
        return {'puntaje': p, 'razon': 'determinista: ' + razon, 'fallback': False}

    ctx = '\n\n'.join(contexto) if contexto else '(ninguno)'
    r, fallback = {}, False
    for clave, q in PREGUNTAS_JUEZ:
        user = (f'PREGUNTA DEL USUARIO:\n{pregunta}\n\nRESPUESTA DE REFERENCIA:\n{referencia}\n\n'
                f'CONTEXTO RECUPERADO:\n{ctx}\n\nRESPUESTA A EVALUAR:\n{respuesta}\n\n'
                f'{q}\nResponde SOLO con SI o NO.')
        salida = generar_juez('Eres un evaluador estricto y objetivo.', user)
        if DEBUG_JUEZ:
            print(f'    [crudo] {clave}: {salida!r}')
        v = _si_no(salida)
        if v is None:                      
            v, fallback = (clave == 'inventa'), True
        r[clave] = v

    if   r['inventa']:        p = 1
    elif not r['correcta']:   p = 2
    elif not r['centrada']:   p = 3
    elif not r['completa']:   p = 4
    else:                     p = 5

    CONTADOR_FALLBACK['total'] += 1
    CONTADOR_FALLBACK['n'] += int(fallback)
    return {'puntaje': p, 'razon': str(r), 'fallback': fallback}


def sim_embeddings(a, b):
    ea, eb = st.encode([a, b])
    return float(np.dot(ea, eb) / (np.linalg.norm(ea) * np.linalg.norm(eb) + 1e-12))


# VARA ORIGINAL DE M2 (rúbrica v1.0), sin cambios, se reporta como columna aparte
PREGUNTAS_JUEZ_M2 = [
 ('no_inventa', '¿La respuesta evita afirmar datos que NO aparecen en la referencia?'),
 ('correcta',   '¿La respuesta coincide en lo esencial con la referencia?'),
 ('centrada',   '¿La respuesta va al punto, sin divagar ni contradecirse?'),
 ('completa',   '¿La respuesta cubre todo lo que se preguntó?'),
]
USAR_JUEZ_M2 = True


def juez_puntua_m2(pregunta, respuesta, referencia):
    """El juez de M2 tal cual: el generador responde cuatro preguntas SI/NO."""
    r = {}
    for clave, q in PREGUNTAS_JUEZ_M2:
        user = (f'PREGUNTA DEL USUARIO:\n{pregunta}\n\nRESPUESTA DE REFERENCIA:\n{referencia}\n\n'
                f'RESPUESTA A EVALUAR:\n{respuesta}\n\n{q}\nResponde SOLO con SI o NO.')
        v = _si_no(generar('Eres un evaluador estricto y objetivo.', user, max_new_tokens=3))
        r[clave] = bool(v)
    if   not r['no_inventa']: return 1
    elif not r['correcta']:   return 2
    elif not r['centrada']:   return 3
    elif not r['completa']:   return 4
    return 5

In [17]:
# Prueba de cordura: el juez debe separar la respuesta real de los dos controles negativos.
DEBUG_JUEZ = True
print('Juez:', JUEZ_DESCRIPCION)
_p = '¿Qué umbral de decisión eligió M1?'
_ref = 'M1 eligió 0,65, maximizando F1 sobre validación.'
prueba_cordura = {}
for etiqueta, texto in [('respuesta correcta', 'M1 eligió el umbral 0,65 maximizando F1 en validación.'),
                        ('respuesta errada  ', 'M1 eligió el umbral 0,20 maximizando accuracy en test.'),
                        ('dato inventado    ', 'M1 eligió 0,65 tras una auditoría externa de Fitch en 2024.')]:
    v = juez_puntua(_p, texto, _ref)
    prueba_cordura[etiqueta.strip()] = v['puntaje']
    print(f'{etiqueta} -> {v["puntaje"]}/5 | {v["razon"]}')
DEBUG_JUEZ = False

JUEZ_VALIDO = (prueba_cordura['respuesta correcta'] >= 4
               and prueba_cordura['respuesta errada'] <= 2
               and prueba_cordura['dato inventado'] <= 1)
print('\nJUEZ_VALIDO:', JUEZ_VALIDO)
if not JUEZ_VALIDO:
    print('El juez no separa la respuesta real de los controles: la dimensión 2 se reporta')
    print('como no válida y la lectura honesta lo declara.')

# La misma prueba para la vara original de M2, para dejar constancia de por qué se complementó.
cordura_m2 = {e: juez_puntua_m2(_p, t, _ref) for e, t in [
    ('respuesta correcta', 'M1 eligió el umbral 0,65 maximizando F1 en validación.'),
    ('respuesta errada', 'M1 eligió el umbral 0,20 maximizando accuracy en test.'),
    ('dato inventado', 'M1 eligió 0,65 tras una auditoría externa de Fitch en 2024.')]}
JUEZ_M2_VALIDO = (cordura_m2['respuesta correcta'] >= 4 and cordura_m2['respuesta errada'] <= 2
                  and cordura_m2['dato inventado'] <= 1)
print(f'\nJuez de M2 ({GEN_MODEL}):', cordura_m2, '-> válido:', JUEZ_M2_VALIDO)

Juez: Qwen/Qwen2.5-7B-Instruct (4bit)
    [crudo] inventa: 'NO'
    [crudo] correcta: 'SI'
    [crudo] centrada: 'SI'
    [crudo] completa: 'SI'
respuesta correcta -> 5/5 | {'inventa': False, 'correcta': True, 'centrada': True, 'completa': True}
    [crudo] inventa: 'NO'
    [crudo] correcta: 'NO'
    [crudo] centrada: 'NO'
    [crudo] completa: 'NO'
respuesta errada   -> 2/5 | {'inventa': False, 'correcta': False, 'centrada': False, 'completa': False}
    [crudo] inventa: 'SI'
    [crudo] correcta: 'NO'
    [crudo] centrada: 'NO'
    [crudo] completa: 'NO'
dato inventado     -> 1/5 | {'inventa': True, 'correcta': False, 'centrada': False, 'completa': False}

JUEZ_VALIDO: True

Juez de M2 (Qwen/Qwen2.5-1.5B-Instruct): {'respuesta correcta': 5, 'respuesta errada': 1, 'dato inventado': 2} -> válido: False


## 8 · Eval set

Quince casos sobre el corpus. Diez tienen la respuesta escrita en los documentos; tres piden una operación sobre cifras del corpus, que es donde la capa agéntica tiene algo que hacer; y dos tienen una respuesta que no está en las fuentes, donde lo correcto es la válvula de escape. Cada caso declara el documento donde está su respuesta, lo que permite medir la recuperación sin juez.

El eval set se congela como `eval_set_m3_v2.json`. No es comparable con el de doce casos de la corrida de prueba.

In [18]:
EVAL_SET_M3 = [
 {'id':'Q01','input':'¿Qué significa el valor -999 en la columna score_credito?',
  'esperado':'Es un valor centinela: un número imposible que significa sin dato. Se convierte a nulo antes de modelar.',
  'criterio':'debe decir que es un centinela que significa sin dato','tipo':'gold','doc_esperado':['doc2']},
 {'id':'Q02','input':'¿Por qué no se imputa el score de buró externo cuando falta?',
  'esperado':'Porque falta más cuando el cliente es peor (MNAR): imputarlo con la mediana borraría la señal.',
  'criterio':'debe mencionar que la ausencia es informativa','tipo':'gold','doc_esperado':['doc2']},
 {'id':'Q03','input':'¿Por qué se excluye la variable sexo del modelo?',
  'esperado':'Porque es una variable protegida y no se usa como predictor.',
  'criterio':'debe decir que es variable protegida','tipo':'gold','doc_esperado':['doc4']},
 {'id':'Q04','input':'¿Qué variables se excluyen por fuga de información?',
  'esperado':'castigos_previos, clv_estimado y propension_churn, porque se calculan después del desenlace.',
  'criterio':'debe nombrar al menos dos y el motivo','tipo':'gold','doc_esperado':['doc4']},
 {'id':'Q05','input':'¿Cuál es el horizonte de la etiqueta y por qué se descartan clientes?',
  'esperado':'12 meses. Se descartan los observados menos de 12 meses porque no se conoce su desenlace.',
  'criterio':'horizonte correcto + motivo del descarte','tipo':'gold','doc_esperado':['doc3']},
 {'id':'Q06','input':'¿Qué PR-AUC reportó BETO con LoRA en test y contra qué baseline se compara?',
  'esperado':'PR-AUC 0,1723 en test, contra TF-IDF con regresión logística que alcanzó 0,1383.',
  'criterio':'debe dar la cifra del modelo afinado','tipo':'gold','doc_esperado':['doc5']},
 {'id':'Q07','input':'Mi modelo da resultados distintos cada vez que lo cargo, ¿por qué puede ser?',
  'esperado':'Porque el pooler de BETO no se guardó con el adaptador y se inicializa al azar en cada carga.',
  'criterio':'consulta coloquial: debe llegar al incidente del pooler','tipo':'gold','doc_esperado':['doc6']},
 {'id':'Q08','input':'¿Entre qué valores varió el ROC-AUC al probar las doce semillas del pooler?',
  'esperado':'Entre 0,2398 y 0,7894 en validación, con el mismo adaptador.',
  'criterio':'debe dar el rango','tipo':'gold','doc_esperado':['doc6']},
 {'id':'Q09','input':'¿A qué grupo perjudica más el modelo según la auditoría?',
  'esperado':'A los informales: 29,3% de falsa alarma frente a 6,0% de los formales. Por estrato, 22,1% en estrato 1 contra 7,4% en estrato 4.',
  'criterio':'debe identificar el grupo y dar las cifras','tipo':'gold','doc_esperado':['doc8']},
 {'id':'Q10','input':'¿Cómo se mitiga el sesgo de posición del juez?',
  'esperado':'Evaluando en ambos órdenes y declarando empate si el veredicto cambia al invertirlos.',
  'criterio':'debe describir la mitigación','tipo':'gold','doc_esperado':['doc7']},
 # --- requieren una operación sobre cifras del corpus: aquí la calculadora tiene algo que hacer
 {'id':'T01','input':'Según la auditoría, ¿cuántas veces mayor es la tasa de falsa alarma en estrato 1 que en estrato 4?',
  'esperado':'Unas 2,99 veces: 22,1% en estrato 1 frente a 7,4% en estrato 4.',
  'criterio':'debe dar 2,99 (22,1 / 7,4)','tipo':'herramienta','doc_esperado':['doc8'],
  'numero': 2.986, 'tol': 0.006},
 {'id':'T02','input':'¿Cuántas columnas del dataset sí entran al modelo?',
  'esperado':'Entran 45 columnas: el dataset tiene 72 y se excluyen 27.',
  'criterio':'debe dar 45 (72 - 27)','tipo':'herramienta','doc_esperado':['doc4'],
  'numero': 45, 'tol': 0},
 {'id':'T03','input':'¿Cuántos puntos porcentuales más incumple la población informal que la formal?',
  'esperado':'1,1 puntos porcentuales: la población informal incumple 4,0% y la formal 2,9%.',
  'criterio':'debe dar 1,1 (4,0 - 2,9)','tipo':'herramienta','doc_esperado':['doc8'],
  'numero': 1.1, 'tol': 0.006},
 # --- fuera del corpus
 {'id':'A01','input':'¿Cuál es la tasa de interés que cobra la entidad por estos créditos?',
  'esperado': VALVULA,
  'criterio':'FUERA DE CORPUS: debe activar la válvula, no inventar una tasa','tipo':'adversarial','doc_esperado':[]},
 {'id':'A02','input':'¿Cuántos empleados tiene el área de riesgo de la entidad?',
  'esperado': VALVULA,
  'criterio':'FUERA DE CORPUS: debe activar la válvula','tipo':'adversarial','doc_esperado':[]},
]

RUTA_EVAL = RUTA_SALIDA / 'eval_set_m3_v2.json'
if RUTA_EVAL.exists():
    with open(RUTA_EVAL, encoding='utf-8') as f:
        eval_set = json.load(f)
    print(f'Eval set congelado cargado ({len(eval_set)} casos).')
else:
    eval_set = EVAL_SET_M3
    with open(RUTA_EVAL, 'w', encoding='utf-8') as f:
        json.dump(eval_set, f, ensure_ascii=False, indent=2)
    print(f'Eval set congelado en {RUTA_EVAL}.')

print(f'{len(eval_set)} casos | '
      + ' | '.join(f'{t}: {sum(c["tipo"] == t for c in eval_set)}' for t in ('gold', 'herramienta', 'adversarial')))

Eval set congelado en m3_salida/eval_set_m3_v2.json.
15 casos | gold: 10 | herramienta: 3 | adversarial: 2


In [19]:
# Transparencia del router: su decisión para cada pregunta del eval set.
_rows = [{'id': c['id'], 'tipo': c['tipo'], 'router: calcular': necesita_calculo(c['input']),
          'necesita (según el eval set)': c['tipo'] == 'herramienta', 'pregunta': c['input'][:70]}
         for c in eval_set]
decisiones_router = pd.DataFrame(_rows).set_index('id')
display(decisiones_router)
_err = decisiones_router[decisiones_router['router: calcular'] != decisiones_router['necesita (según el eval set)']]
print(f'Errores del router en el eval set: {len(_err)}', list(_err.index) or '—')

,tipo,router: calcular,necesita (según el eval set),pregunta
id,,,,
Q01,gold,False,False,¿Qué significa el valor -999 en la columna score_credito?
Q02,gold,False,False,¿Por qué no se imputa el score de buró externo cuando falta?
Q03,gold,False,False,¿Por qué se excluye la variable sexo del modelo?
Q04,gold,False,False,¿Qué variables se excluyen por fuga de información?
Q05,gold,False,False,¿Cuál es el horizonte de la etiqueta y por qué se descartan clientes?
Q06,gold,False,False,¿Qué PR-AUC reportó BETO con LoRA en test y contra qué baseline se...
Q07,gold,False,False,"Mi modelo da resultados distintos cada vez que lo cargo, ¿por qué ..."
Q08,gold,False,False,¿Entre qué valores varió el ROC-AUC al probar las doce semillas de...
Q09,gold,False,False,¿A qué grupo perjudica más el modelo según la auditoría?


Errores del router en el eval set: 0 —


In [ ]:
UMBRAL_SIM = 0.60     


def _numeros(texto):
    return [float(x.replace(',', '.')) for x in re.findall(r'\d+(?:[.,]\d+)?', str(texto))]


def acierto_dominio(caso, respuesta, sim, puntaje):
    """Criterio por caso. En los adversariales no basta acertar: hay que abstenerse.
    En los de herramienta el criterio es la cifra, verificable sin juez."""
    if caso['tipo'] == 'adversarial':
        activo = es_valvula(respuesta)
        return activo, ('activó la válvula' if activo else 'NO se abstuvo: respondió igual')
    if caso['tipo'] == 'herramienta':
        ok = any(abs(x - caso['numero']) <= caso['tol'] for x in _numeros(respuesta))
        return ok, f"{'da' if ok else 'NO da'} {caso['numero']}"
    ok = (sim >= UMBRAL_SIM) or (puntaje >= 4)
    return ok, f'sim {sim:.2f} / juez {puntaje}'


def _desempacar(salida):
    """Acepta respuesta, (respuesta, pasajes) o (respuesta, pasajes, llamadas)."""
    if isinstance(salida, tuple):
        return salida if len(salida) == 3 else (salida[0], salida[1], [])
    return salida, [], []


def tipo_falla(caso, ok, resp, doc_ok, hay_contexto):
    """Recuperación si el documento correcto no llegó al contexto; generación si llegó."""
    if ok:
        return ''
    if caso['tipo'] == 'adversarial':
        return 'no se abstuvo'
    if not hay_contexto:
        return 'sin recuperación'
    if not doc_ok:
        return 'recuperación'
    return 'generación (válvula de más)' if es_valvula(resp) else 'generación'


def harness(eval_set, sistema, nombre='sistema'):
    acepta_ctx = 'devolver_contexto' in inspect.signature(sistema).parameters
    detalle = []
    for c in eval_set:
        t0 = time.perf_counter()
        salida = sistema(c['input'], devolver_contexto=True) if acepta_ctx else sistema(c['input'])
        seg = time.perf_counter() - t0
        resp, pasajes, llamadas = _desempacar(salida)

        ctx_textos = [p['texto'] for p in pasajes]
        obs = [f"Resultado de {l['tool']}({json.dumps(l['args'], ensure_ascii=False)}): {l['obs']}"
               for l in llamadas]
        docs_rec = {p['id'].split('_')[0] for p in pasajes}
        doc_ok = (bool(set(c['doc_esperado']) & docs_rec)
                  if pasajes and c.get('doc_esperado') else None)

        sim = sim_embeddings(resp, c['esperado'])
        vj = juez_puntua(c['input'], resp, c['esperado'], ctx_textos + obs)
        j_m2 = juez_puntua_m2(c['input'], resp, c['esperado']) if USAR_JUEZ_M2 else float('nan')
        ok, nota = acierto_dominio(c, resp, sim, vj['puntaje'])

        detalle.append({'id': c['id'], 'tipo': c['tipo'], 'sim': round(sim, 3),
                        'juez': vj['puntaje'], 'juez_m2': j_m2, 'acierto': ok, 'nota': nota,
                        'tipo_falla': tipo_falla(c, ok, resp, doc_ok, bool(pasajes)),
                        'doc_recuperado': doc_ok,
                        'herramientas': [l['tool'] for l in llamadas], 'n_llamadas': len(llamadas),
                        'segundos': round(seg, 2), 'respuesta': resp, 'juez_razon': vj['razon'],
                        'contexto': [p['id'] for p in pasajes],
                        'contexto_texto': ctx_textos, 'obs_herramientas': obs})

    d = pd.DataFrame(detalle)
    gold, herr, adv = d[d.tipo == 'gold'], d[d.tipo == 'herramienta'], d[d.tipo == 'adversarial']
    rec = d.doc_recuperado.dropna().astype(bool)
    return {'sistema': nombre,
            'sim_promedio': float(d[d.tipo != 'adversarial'].sim.mean()),   # sin la válvula, que copia el texto esperado
            'juez_promedio': float(d.juez.mean()),
            'juez_m2_promedio': float(d.juez_m2.mean()),
            'aciertos': int(d.acierto.sum()), 'total': len(d),
            'aciertos_gold': f'{int(gold.acierto.sum())}/{len(gold)}',
            'aciertos_herramienta': f'{int(herr.acierto.sum())}/{len(herr)}',
            'valvula': f'{int(adv.acierto.sum())}/{len(adv)}',
            'recuperacion': f'{int(rec.sum())}/{len(rec)}' if len(rec) else '—',
            'llamadas': int(d.n_llamadas.sum()),
            'casos_T_con_tool': f'{int((herr.n_llamadas > 0).sum())}/{len(herr)}',
            'llamadas_de_mas': int(d[d.tipo != 'herramienta'].n_llamadas.sum()),
            'latencia_mediana': float(d.segundos.median()),
            'detalle': d}

## 9 · Tabla evolutiva

In [21]:
SISTEMAS = [
 ('Línea base M2 (sin RAG)',      sistema_sin_rag),
 ('RAG ingenuo (S07)',            sistema_rag),
 ('RAG avanzado (híbrida+rerank)', sistema_rag_avanzado),
 ('Sistema final (con herramienta)', sistema_final),
]

CONTADOR_FALLBACK.update(n=0, total=0, deterministas=0)
resultados = {}
for nombre, fn in SISTEMAS:
    print(f'corriendo: {nombre} ...', flush=True)
    resultados[nombre] = harness(eval_set, fn, nombre)

tabla = pd.DataFrame([{
    'Sistema': r['sistema'],
    'D1 · similitud (sin válvula)': round(r['sim_promedio'], 3),
    'D2 · juez v1.1 (1-5)': round(r['juez_promedio'], 2),
    'D2 · juez M2 (1-5)': round(r['juez_m2_promedio'], 2),
    'D3 · aciertos': f"{r['aciertos']}/{r['total']}",
    'gold': r['aciertos_gold'],
    'herramienta': r['aciertos_herramienta'],
    'válvula': r['valvula'],
    'doc correcto recuperado': r['recuperacion'],
    'casos T con tool': r['casos_T_con_tool'],
    'tools de más': r['llamadas_de_mas'],
    'latencia mediana (s)': round(r['latencia_mediana'], 2),
} for r in resultados.values()]).set_index('Sistema')

display(tabla)
print(f'Juez 7B: {JUEZ_DESCRIPCION} | prueba de cordura: {"superada" if JUEZ_VALIDO else "FALLIDA"}')
print(f'Juez M2: {GEN_MODEL} | prueba de cordura: {"superada" if JUEZ_M2_VALIDO else "FALLIDA"}')
print('"tools de más" = llamadas a herramientas en casos gold o adversariales, que no las necesitan.')
print(f'Llamadas al juez sin veredicto legible: {CONTADOR_FALLBACK["n"]}/{CONTADOR_FALLBACK["total"]} | '
      f'casos calificados con la regla de la válvula: {CONTADOR_FALLBACK["deterministas"]}')

corriendo: Línea base M2 (sin RAG) ...
corriendo: RAG ingenuo (S07) ...
corriendo: RAG avanzado (híbrida+rerank) ...
corriendo: Sistema final (con herramienta) ...


,D1 · similitud (sin válvula),D2 · juez v1.1 (1-5),D2 · juez M2 (1-5),D3 · aciertos,gold,herramienta,válvula,doc correcto recuperado,casos T con tool,tools de más,latencia mediana (s)
Sistema,,,,,,,,,,,
Línea base M2 (sin RAG),0.410,2.0,1.73,2/15,2/10,0/3,0/2,—,0/3,0,2.26
RAG ingenuo (S07),0.615,4.4,3.53,12/15,8/10,2/3,2/2,12/13,0/3,0,1.70
RAG avanzado (híbrida+rerank),0.645,4.2,2.93,12/15,9/10,1/3,2/2,13/13,0/3,0,1.36
Sistema final (con herramienta),0.651,4.2,2.93,12/15,9/10,1/3,2/2,13/13,1/3,0,1.77


Juez 7B: Qwen/Qwen2.5-7B-Instruct (4bit) | prueba de cordura: superada
Juez M2: Qwen/Qwen2.5-1.5B-Instruct | prueba de cordura: FALLIDA
"tools de más" = llamadas a herramientas en casos gold o adversariales, que no las necesitan.
Llamadas al juez sin veredicto legible: 0/49 | casos calificados con la regla de la válvula: 11


In [22]:
final = resultados['Sistema final (con herramienta)']['detalle']
display(final[['id', 'tipo', 'sim', 'juez', 'acierto', 'nota', 'tipo_falla', 'herramientas', 'segundos']])

fallidos = final[~final.acierto]
print(f'\nCasos fallidos del sistema final: {len(fallidos)}/{len(final)}\n')
for _, f in fallidos.iterrows():
    print(f"{f.id} ({f.nota} · {f.tipo_falla}) · contexto {f.contexto} · herramientas {f.herramientas}")
    print(f'   {f.respuesta[:220]}\n')

,id,tipo,sim,juez,acierto,nota,tipo_falla,herramientas,segundos
0,Q01,gold,0.533,5,True,sim 0.53 / juez 5,,[],1.36
1,Q02,gold,0.037,2,False,sim 0.04 / juez 2,generación (válvula de más),[],0.63
2,Q03,gold,0.620,5,True,sim 0.62 / juez 5,,[],0.92
3,Q04,gold,0.721,5,True,sim 0.72 / juez 5,,[],1.28
4,Q05,gold,0.756,5,True,sim 0.76 / juez 5,,[],2.34
5,Q06,gold,0.840,2,True,sim 0.84 / juez 2,,[],3.65
6,Q07,gold,0.530,5,True,sim 0.53 / juez 5,,[],4.14
7,Q08,gold,0.501,5,True,sim 0.50 / juez 5,,[],1.77
8,Q09,gold,0.791,5,True,sim 0.79 / juez 5,,[],6.34
9,Q10,gold,0.851,5,True,sim 0.85 / juez 5,,[],1.68



Casos fallidos del sistema final: 3/15

Q02 (sim 0.04 / juez 2 · generación (válvula de más)) · contexto ['doc2_c2', 'doc8_c1', 'doc6_c0', 'doc6_c1'] · herramientas []
   No tengo esa información en mis fuentes.

T01 (NO da 2.986 · generación) · contexto ['doc8_c0', 'doc3_c1', 'doc6_c1', 'doc8_c2'] · herramientas ['calculadora']
   Según la auditoría, la tasa de falsa alarma en estrato 1 es de 22,1 por ciento, mientras que en estrato 4 es de 7,4 por ciento. Por lo tanto, la diferencia es de 22,1 - 7,4 = 14,7 puntos porcentuales.

T02 (NO da 45 · generación) · contexto ['doc4_c0', 'doc1_c0', 'doc5_c0', 'doc1_c2'] · herramientas []
   Según el contexto proporcionado, hay 49 columnas que entran al modelo.



In [ ]:
corto = {'Línea base M2 (sin RAG)': 'base', 'RAG ingenuo (S07)': 'ingenuo',
         'RAG avanzado (híbrida+rerank)': 'avanzado', 'Sistema final (con herramienta)': 'final'}
por_sistema = {n: r['detalle'].set_index('id') for n, r in resultados.items()}
nombres = [n for n, _ in SISTEMAS]

caso_a_caso = pd.DataFrame({'tipo': por_sistema[nombres[0]]['tipo']})
for n in nombres:
    d, s = por_sistema[n], corto.get(n, n)
    caso_a_caso[f'{s} ok'] = d['acierto'].map({True: '✓', False: '✗'})
    caso_a_caso[f'{s} juez'] = d['juez']
    caso_a_caso[f'{s} falla'] = d['tipo_falla']
display(caso_a_caso)

print('Cambios de acierto entre sistemas consecutivos:')
for a, b in zip(nombres, nombres[1:]):
    da, db = por_sistema[a]['acierto'].astype(bool), por_sistema[b]['acierto'].astype(bool)
    gana, pierde = list(da[~da & db].index), list(da[da & ~db].index)
    print(f'  {corto[a]:>8} -> {corto[b]:<8}  gana {gana or "—"}  pierde {pierde or "—"}')

,tipo,base ok,base juez,base falla,ingenuo ok,ingenuo juez,ingenuo falla,avanzado ok,avanzado juez,avanzado falla,final ok,final juez,final falla
id,,,,,,,,,,,,,
Q01,gold,✓,5,,✓,5,,✓,5,,✓,5,
Q02,gold,✗,2,sin recuperación,✗,2,generación (válvula de más),✗,2,generación (válvula de más),✗,2,generación (válvula de más)
Q03,gold,✗,2,sin recuperación,✓,5,,✓,5,,✓,5,
Q04,gold,✗,2,sin recuperación,✓,5,,✓,5,,✓,5,
Q05,gold,✗,2,sin recuperación,✓,5,,✓,5,,✓,5,
Q06,gold,✓,2,,✓,5,,✓,2,,✓,2,
Q07,gold,✗,2,sin recuperación,✗,2,recuperación,✓,5,,✓,5,
Q08,gold,✗,2,sin recuperación,✓,5,,✓,5,,✓,5,
Q09,gold,✗,2,sin recuperación,✓,5,,✓,5,,✓,5,


Cambios de acierto entre sistemas consecutivos:
      base -> ingenuo   gana ['Q03', 'Q04', 'Q05', 'Q08', 'Q09', 'Q10', 'T02', 'T03', 'A01', 'A02']  pierde —
   ingenuo -> avanzado  gana ['Q07']  pierde ['T02']
  avanzado -> final     gana —  pierde —


### Ablación: una técnica a la vez

Comparamos las versiones intermedias con el mismo prompt, generador, harness y eval set para separar el aporte de BM25 con RRF y del reranker.

Para la capa agéntica comparamos el RAG avanzado, el agente libre y el sistema final con router. La medición de recuperación por separado permite distinguir el tiempo de búsqueda del tiempo de generación, que representa la mayor parte de la latencia total.

In [ ]:
def sistema_rag_hibrida(pregunta, k=TOP_K, devolver_contexto=False):
    """Ablación: híbrida BM25 + densa con RRF, sin reranking."""
    pasajes = buscar_hibrida(pregunta, k=k)
    r = generar(SYSTEM_RAG, f'Contexto:\n{_contexto(pasajes)}\n\nPregunta: {pregunta}')
    return (r, pasajes) if devolver_contexto else r


def sistema_rag_rerank(pregunta, k=TOP_K, devolver_contexto=False):
    """Ablación: densa top-10 + reranking, sin BM25."""
    pasajes = rerankear(pregunta, buscar_densa(pregunta, 10), k=k)
    r = generar(SYSTEM_RAG, f'Contexto:\n{_contexto(pasajes)}\n\nPregunta: {pregunta}')
    return (r, pasajes) if devolver_contexto else r


CORRER_ABLACION = True
ABLACION = [('RAG ingenuo (S07)', None),
            ('+ híbrida, sin rerank', sistema_rag_hibrida),
            ('+ rerank, sin BM25', sistema_rag_rerank),
            ('RAG avanzado (híbrida+rerank)', None)]

tabla_ablacion, resultados_ablacion = None, {}
if CORRER_ABLACION:
    for nombre, fn in ABLACION:
        if fn is not None:
            print(f'corriendo: {nombre} ...', flush=True)
            resultados_ablacion[nombre] = harness(eval_set, fn, nombre)
    todos = {**resultados, **resultados_ablacion}
    tabla_ablacion = pd.DataFrame([{
        'Sistema': n,
        'D1 · similitud (sin válvula)': round(todos[n]['sim_promedio'], 3),
        'D2 · juez v1.1 (1-5)': round(todos[n]['juez_promedio'], 2),
        'D2 · juez M2 (1-5)': round(todos[n]['juez_m2_promedio'], 2),
        'D3 · aciertos': f"{todos[n]['aciertos']}/{todos[n]['total']}",
        'doc correcto recuperado': todos[n]['recuperacion'],
        'latencia mediana (s)': round(todos[n]['latencia_mediana'], 2),
    } for n, _ in ABLACION]).set_index('Sistema')
    display(tabla_ablacion)

    base_ab = todos['RAG ingenuo (S07)']['detalle'].set_index('id')['acierto'].astype(bool)
    print('Frente al RAG ingenuo:')
    for n, _ in ABLACION[1:]:
        x = todos[n]['detalle'].set_index('id')['acierto'].astype(bool)
        print(f'  {n:<32} gana {list(x[~base_ab & x].index) or "—"}  pierde {list(x[base_ab & ~x].index) or "—"}')

# Capa agéntica
CORRER_AGENTE_LIBRE = True
tabla_agente = None
if CORRER_AGENTE_LIBRE:
    print('\ncorriendo: Agente libre (el modelo decide) ...', flush=True)
    resultados_ablacion['Agente libre (el modelo decide)'] = harness(eval_set, sistema_agente_libre,
                                                                     'Agente libre (el modelo decide)')
    todos = {**resultados, **resultados_ablacion}
    FILAS_AGENTE = ['RAG avanzado (híbrida+rerank)', 'Agente libre (el modelo decide)',
                    'Sistema final (con herramienta)']
    tabla_agente = pd.DataFrame([{
        'Sistema': n,
        'D2 · juez v1.1 (1-5)': round(todos[n]['juez_promedio'], 2),
        'D3 · aciertos': f"{todos[n]['aciertos']}/{todos[n]['total']}",
        'gold': todos[n]['aciertos_gold'],
        'herramienta': todos[n]['aciertos_herramienta'],
        'válvula': todos[n]['valvula'],
        'casos T con tool': todos[n]['casos_T_con_tool'],
        'tools de más': todos[n]['llamadas_de_mas'],
        'latencia mediana (s)': round(todos[n]['latencia_mediana'], 2),
    } for n in FILAS_AGENTE]).set_index('Sistema')
    display(tabla_agente)
    base_av = todos[FILAS_AGENTE[0]]['detalle'].set_index('id')['acierto'].astype(bool)
    print('Frente al RAG avanzado:')
    for n in FILAS_AGENTE[1:]:
        x = todos[n]['detalle'].set_index('id')['acierto'].astype(bool)
        print(f'  {n:<34} gana {list(x[~base_av & x].index) or "—"}  pierde {list(x[base_av & ~x].index) or "—"}')

corriendo: + híbrida, sin rerank ...
corriendo: + rerank, sin BM25 ...


,D1 · similitud (sin válvula),D2 · juez v1.1 (1-5),D2 · juez M2 (1-5),D3 · aciertos,doc correcto recuperado,latencia mediana (s)
Sistema,,,,,,
RAG ingenuo (S07),0.615,4.4,3.53,12/15,12/13,1.70
"+ híbrida, sin rerank",0.688,4.4,3.67,12/15,12/13,2.03
"+ rerank, sin BM25",0.683,4.0,2.87,12/15,13/13,1.85
RAG avanzado (híbrida+rerank),0.645,4.2,2.93,12/15,13/13,1.36


Frente al RAG ingenuo:
  + híbrida, sin rerank            gana ['Q02']  pierde ['T02']
  + rerank, sin BM25               gana ['Q02']  pierde ['T02']
  RAG avanzado (híbrida+rerank)    gana ['Q07']  pierde ['T02']

corriendo: Agente libre (el modelo decide) ...


,D2 · juez v1.1 (1-5),D3 · aciertos,gold,herramienta,válvula,casos T con tool,tools de más,latencia mediana (s)
Sistema,,,,,,,,
RAG avanzado (híbrida+rerank),4.2,12/15,9/10,1/3,2/2,0/3,0,1.36
Agente libre (el modelo decide),4.0,9/15,6/10,1/3,2/2,2/3,4,2.93
Sistema final (con herramienta),4.2,12/15,9/10,1/3,2/2,1/3,0,1.77


Frente al RAG avanzado:
  Agente libre (el modelo decide)    gana ['T02']  pierde ['Q01', 'Q07', 'Q09', 'T03']
  Sistema final (con herramienta)    gana —  pierde —


In [ ]:
# Costo propio de cada técnica (latencia de la recuperación sola sin generar, sobre las preguntas del eval set)
def _mediana_ms(fn, repeticiones=3):
    ts = []
    for c in eval_set:
        for _ in range(repeticiones):
            t0 = time.perf_counter(); fn(c['input']); ts.append(time.perf_counter() - t0)
    return round(1000 * float(np.median(ts)), 1)

latencia_recuperacion = pd.DataFrame([{'recuperación': n, 'mediana (ms)': _mediana_ms(f)} for n, f in [
    ('densa top-4',      lambda q: buscar_densa(q, TOP_K)),
    ('híbrida top-4',    lambda q: buscar_hibrida(q, k=TOP_K)),
    ('densa + rerank',   lambda q: rerankear(q, buscar_densa(q, 10))),
    ('híbrida + rerank', lambda q: recuperar_avanzado(q, TOP_K)),
]]).set_index('recuperación')
display(latencia_recuperacion)

,mediana (ms)
recuperación,
densa top-4,12.4
híbrida top-4,13.1
densa + rerank,46.8
híbrida + rerank,46.6


## 10 · RAGAS

Calculamos las cuatro métricas del laboratorio S10 para el RAG ingenuo y el sistema final. Excluimos los casos fuera del corpus: con la válvula, `context_recall` es cero por diseño y `answer_relevancy` no tiene sentido. Y en el sistema final los resultados de las herramientas cuentan como contexto para `faithfulness`, porque una cifra calculada no está escrita en ningún chunk.

Leemos RAGAS junto con la medida determinista de documento correcto recuperado. En la corrida de prueba hubo discrepancias entre ambas, Q02 tenía el chunk correcto en primer lugar en ambos sistemas y su `context_recall` dio 1 en uno y 0 en el otro; así que no usamos las métricas del juez como única evidencia de recuperación.

In [ ]:
def _si(t):
    return 1 if _si_no(t) else 0


def faithfulness(caso):
    """¿Cada afirmación de la respuesta está respaldada por el contexto (o por una herramienta)?"""
    ctx = '\n'.join(list(caso['contexto_texto']) + list(caso['obs']))
    afirm = [s.strip() for s in re.split(r'(?<!\d)[.\n]|[.\n](?!\d)', caso['respuesta']) if len(s.strip()) > 15]
    if not ctx or not afirm or es_valvula(caso['respuesta']):
        return float('nan')
    ok = sum(_si(generar_juez('Responde solo SI o NO.',
             f'Contexto:\n{ctx}\n\n¿El contexto respalda esta afirmación? "{a}"', 3)) for a in afirm)
    return ok / len(afirm)


def context_precision(caso):
    """De los chunks recuperados, ¿cuántos son relevantes?"""
    if not caso['contexto_texto']:
        return float('nan')
    ok = sum(_si(generar_juez('Responde solo SI o NO.',
             f'Pregunta: {caso["input"]}\n\n¿Este pasaje sirve para responderla? "{ch}"', 3))
             for ch in caso['contexto_texto'])
    return ok / len(caso['contexto_texto'])


def context_recall(caso):
    """¿El contexto contiene lo necesario para llegar a la referencia?"""
    ctx = '\n'.join(caso['contexto_texto'])
    if not ctx:
        return float('nan')
    return float(_si(generar_juez('Responde solo SI o NO.',
        f'Contexto:\n{ctx}\n\n¿El contexto contiene lo necesario para llegar a esta '
        f'referencia? "{caso["esperado"]}"', 3)))


def answer_relevancy(caso):
    """¿La respuesta contesta la pregunta que se hizo? Una abstención no la contesta: 0."""
    if es_valvula(caso['respuesta']):
        return 0.0
    q2 = generar_juez('Genera SOLO la pregunta que esta respuesta contestaría, sin nada más.',
                      caso['respuesta'], 40)
    return sim_embeddings(caso['input'], q2)


def ragas(sc, nombre):
    det = sc['detalle'].set_index('id')
    filas = []
    for c in eval_set:
        if c['tipo'] == 'adversarial':
            continue
        caso = {'input': c['input'], 'esperado': c['esperado'],
                'respuesta': det.loc[c['id'], 'respuesta'],
                'contexto_texto': det.loc[c['id'], 'contexto_texto'],
                'obs': det.loc[c['id'], 'obs_herramientas']}
        filas.append({'id': c['id'], 'tipo': c['tipo'],
                      'faithfulness': faithfulness(caso),
                      'context_precision': context_precision(caso),
                      'context_recall': context_recall(caso),
                      'answer_relevancy': answer_relevancy(caso),
                      'doc_recuperado (determinista)': det.loc[c['id'], 'doc_recuperado']})
    d = pd.DataFrame(filas)
    d.insert(0, 'sistema', nombre)
    return d


CORRER_RAGAS_INGENUO = True
ragas_final = ragas(resultados['Sistema final (con herramienta)'], 'Sistema final')
partes = [ragas_final]
if CORRER_RAGAS_INGENUO:
    partes.append(ragas(resultados['RAG ingenuo (S07)'], 'RAG ingenuo'))
ragas_detalle = pd.concat(partes, ignore_index=True)

MET = ['faithfulness', 'context_precision', 'context_recall', 'answer_relevancy']
tabla_ragas = ragas_detalle.groupby('sistema')[MET].mean().round(3)
display(tabla_ragas)
display(ragas_detalle.groupby('sistema')[MET].count().rename(columns=lambda c: c + ' (n)'))

# context_recall = 1 debería coincidir con doc recuperado.
_cmp = ragas_detalle.dropna(subset=['context_recall', 'doc_recuperado (determinista)'])
_disc = _cmp[_cmp['context_recall'].astype(bool) != _cmp['doc_recuperado (determinista)'].astype(bool)]
print(f'context_recall discrepa de la recuperación determinista en {len(_disc)}/{len(_cmp)} casos:',
      list(zip(_disc.sistema, _disc.id)) or '—')
print('Juez de RAGAS:', JUEZ_DESCRIPCION, '· embeddings:', EMB_MODEL)

,faithfulness,context_precision,context_recall,answer_relevancy
sistema,,,,
RAG ingenuo,0.778,0.288,0.923,0.727
Sistema final,0.792,0.308,0.923,0.719


,faithfulness (n),context_precision (n),context_recall (n),answer_relevancy (n)
sistema,,,,
RAG ingenuo,12,13,13,13
Sistema final,12,13,13,13


context_recall discrepa de la recuperación determinista en 1/26 casos: [('Sistema final', 'T01')]
Juez de RAGAS: Qwen/Qwen2.5-7B-Instruct (4bit) · embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


## 11 · Archivos de salida y borrador del README

In [ ]:
import sys, importlib.metadata as _im

def _v(p):
    try:
        return _im.version(p)
    except Exception:
        return 'no instalado'

VERSIONES = {'python': sys.version.split()[0],
             **{p: _v(p) for p in ['torch', 'transformers', 'sentence-transformers', 'chromadb', 'rank-bm25',
                                   'bitsandbytes', 'accelerate', 'pandas', 'numpy']},
             'generador': GEN_MODEL, 'juez': JUEZ_DESCRIPCION, 'embeddings': EMB_MODEL,
             'reranker': RERANK_MODEL, 'semilla': SEMILLA, 'decodificacion': 'greedy (do_sample=False)'}

tabla.to_csv(RUTA_SALIDA / 'scorecard_m3.csv', encoding='utf-8')
tabla_ragas.to_csv(RUTA_SALIDA / 'ragas_m3.csv', encoding='utf-8')
ragas_detalle.to_csv(RUTA_SALIDA / 'ragas_m3_detalle.csv', index=False, encoding='utf-8')
diag.to_csv(RUTA_SALIDA / 'consultas_diagnostico.csv', index=False, encoding='utf-8')
rangos_diag.to_csv(RUTA_SALIDA / 'rangos_diagnostico.csv', index=False, encoding='utf-8')
caso_a_caso.to_csv(RUTA_SALIDA / 'caso_a_caso.csv', encoding='utf-8')
latencia_recuperacion.to_csv(RUTA_SALIDA / 'latencia_recuperacion.csv', encoding='utf-8')
pd.DataFrame(demos_agente).to_csv(RUTA_SALIDA / 'demos_agente.csv', index=False, encoding='utf-8')
if tabla_ablacion is not None:
    tabla_ablacion.to_csv(RUTA_SALIDA / 'ablacion_m3.csv', encoding='utf-8')
if tabla_agente is not None:
    tabla_agente.to_csv(RUTA_SALIDA / 'ablacion_agente_m3.csv', encoding='utf-8')
decisiones_router.to_csv(RUTA_SALIDA / 'decisiones_router.csv', encoding='utf-8')

pd.concat([r['detalle'].assign(sistema=n) for n, r in {**resultados, **resultados_ablacion}.items()]) \
  .drop(columns=['contexto_texto']) \
  .to_csv(RUTA_SALIDA / 'scorecard_m3_detalle.csv', index=False, encoding='utf-8')

with open(RUTA_SALIDA / 'versiones.json', 'w', encoding='utf-8') as f:
    json.dump(VERSIONES, f, ensure_ascii=False, indent=2)

with open(RUTA_SALIDA / 'RUBRICA_M3.md', 'w', encoding='utf-8') as f:
    f.write('# Rúbrica del juez · M3 v1.1\n\n')
    f.write(f'Juez: `{JUEZ_DESCRIPCION}`, greedy, distinto del generador `{GEN_MODEL}`'
            f'{" (no se pudo cargar: el juez ES el generador)" if JUEZ_ES_GENERADOR else ""}.\n'
            f'Rúbrica multi-criterio: el nivel 1-5 se deriva de cuatro preguntas binarias respondidas '
            f'con SI/NO (`max_new_tokens=3`), con el contexto recuperado a la vista. Los casos de '
            f'válvula se califican con una regla determinista.\n'
            f'Prueba de cordura superada: **{JUEZ_VALIDO}**. '
            f'La vara original de M2 (juez `{GEN_MODEL}`, rúbrica v1.0) se reporta como columna aparte; '
            f'su prueba de cordura: **{JUEZ_M2_VALIDO}**.\n\n')
    f.write(f'```\n{RUBRICA}\n```\n')

with open(RUTA_SALIDA / 'corpus_m3.json', 'w', encoding='utf-8') as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)


def _tabla_md(df):
    df = df.reset_index()
    cab = '| ' + ' | '.join(map(str, df.columns)) + ' |'
    sep = '| ' + ' | '.join(['---'] * len(df.columns)) + ' |'
    filas = ['| ' + ' | '.join(str(x) for x in fila) + ' |' for fila in df.itertuples(index=False)]
    return '\n'.join([cab, sep, *filas])

DIAGRAMA = """```mermaid
flowchart LR
  D[Documentos del proyecto] --> C[Chunks de 500 con 80 de solapamiento]
  C --> E[Embeddings MiniLM multilingüe] --> V[(Chroma)]
  C --> B[(Índice BM25)]
  Q[Pregunta] --> DS[Densa top-10]
  Q --> BS[BM25 top-10]
  V -.-> DS
  B -.-> BS
  DS --> R[Fusión RRF]:::adv
  BS --> R
  R --> X[Cross-encoder, top-4]:::adv
  X --> RT{Router: ¿pide una operación?}:::tool
  RT -->|no| P[Prompt base con válvula + contexto]
  RT -->|sí| EX[Qwen propone la expresión]:::tool
  EX --> T[Calculadora]:::tool
  T -->|resultado como pasaje| P
  P --> G[Qwen2.5-1.5B]
  G --> A[Respuesta con fuente, o la válvula]
  classDef adv fill:#dbeafe,stroke:#1d4ed8
  classDef tool fill:#fef3c7,stroke:#b45309
```"""

n_tipo = {t: sum(c['tipo'] == t for c in eval_set) for t in ('gold', 'herramienta', 'adversarial')}
readme = f"""# SI4006 · M3 — RAG agéntico sobre el sistema de riesgo crediticio

> Documento generado a partir de la ejecución del notebook. Las tablas y métricas corresponden a los resultados obtenidos en esta corrida.

**Una herramienta que ayuda a consultar información sobre solicitudes de crédito, analizar los datos de cada persona y responder preguntas con base en la información disponible. También puede realizar cálculos cuando se necesitan.**

Integrantes: {', '.join(INTEGRANTES)} · Repositorio: {REPO_URL}

## Pipeline

{DIAGRAMA}

Técnicas avanzadas en azul; capa agéntica en amarillo.

## Corpus e índice

{len(corpus)} documentos ({sum(len(d['texto']) for d in corpus):,} caracteres), {len(chunks)} chunks de
{CHUNK_SIZE} caracteres con {OVERLAP} de solapamiento, top-k = {TOP_K}. Fuente: documentación propia del
equipo sobre M1 y M2, de uso académico (`m3_salida/corpus_m3.json`). Las tres decisiones están
justificadas en la sección 2 del notebook.

## Tabla evolutiva

Eval set de {len(eval_set)} casos: {n_tipo['gold']} con la respuesta en el corpus, {n_tipo['herramienta']} que
piden una operación y {n_tipo['adversarial']} fuera del corpus. Juez: `{JUEZ_DESCRIPCION}` (prueba de
cordura: {'superada' if JUEZ_VALIDO else 'FALLIDA'}). La vara original de M2 se reporta en
`D2 · juez M2` (prueba de cordura: {'superada' if JUEZ_M2_VALIDO else 'FALLIDA'}).

{_tabla_md(tabla)}

## Ablación

{_tabla_md(tabla_ablacion) if tabla_ablacion is not None else '_No se corrió._'}

Latencia de la recuperación sola:

{_tabla_md(latencia_recuperacion)}

## RAGAS

Implementación a mano del laboratorio de S10, con `{JUEZ_DESCRIPCION}` como juez y
`{EMB_MODEL}` para answer relevancy. Sin los casos fuera del corpus.

{_tabla_md(tabla_ragas)}

## Capa agéntica

Un router con reglas decide si la pregunta pide una operación; el generador propone la expresión y
el sistema llama la calculadora. Quién decide la herramienta, medido sobre el mismo eval set:

{_tabla_md(tabla_agente) if tabla_agente is not None else '_No se corrió._'}

Demostraciones (fuera del eval set):

{_tabla_md(pd.DataFrame(demos_agente)[['consulta', 'necesita', 'llamadas', 'expresion', 'veredicto']].set_index('consulta'))}

## Lectura honesta

1. ¿Qué movió qué?
2. ¿Cuánto costó?
3. ¿Qué falla queda y de qué tipo es?
4. ¿Qué pasó con las preguntas cuya respuesta no está en el corpus?

## Reproducibilidad

Semilla {SEMILLA}, decodificación greedy. Versiones en `m3_salida/versiones.json`. El notebook corre
de principio a fin en Colab gratuito con GPU T4, sin claves de pago.

## Uso de IA

_[escribir: qué herramientas usaron, para qué partes (código, redacción, depuración, análisis) y qué
revisaron o cambiaron ustedes]_
"""
with open(RUTA_SALIDA / 'README_borrador.md', 'w', encoding='utf-8') as f:
    f.write(readme)

print('Escrito en', RUTA_SALIDA.resolve())
for p in sorted(RUTA_SALIDA.iterdir()):
    print(' -', p.name)

Escrito en /content/m3_salida
 - README_borrador.md
 - RUBRICA_M3.md
 - ablacion_agente_m3.csv
 - ablacion_m3.csv
 - caso_a_caso.csv
 - consultas_diagnostico.csv
 - corpus_m3.json
 - decisiones_router.csv
 - demos_agente.csv
 - eval_set_m3_v2.json
 - latencia_recuperacion.csv
 - ragas_m3.csv
 - ragas_m3_detalle.csv
 - rangos_diagnostico.csv
 - scorecard_m3.csv
 - scorecard_m3_detalle.csv
 - versiones.json


In [28]:
print('=' * 78)
print('RESUMEN M3'.center(78))
print('=' * 78)
print(f'Corpus            : {len(corpus)} documentos, {len(chunks)} chunks '
      f'(size {CHUNK_SIZE}, overlap {OVERLAP}, top-k {TOP_K})')
print(f'Eval set          : {len(eval_set)} casos | '
      + ', '.join(f'{sum(c["tipo"] == t for c in eval_set)} {t}' for t in ('gold', 'herramienta', 'adversarial')))
print(f'Generador         : {GEN_MODEL}')
print(f'Juez v1.1         : {JUEZ_DESCRIPCION}{"  <- ES EL GENERADOR" if JUEZ_ES_GENERADOR else ""} | '
      f'prueba de cordura: {"superada" if JUEZ_VALIDO else "FALLIDA"}')
print(f'Juez M2 (original): {GEN_MODEL} | prueba de cordura: {"superada" if JUEZ_M2_VALIDO else "FALLIDA"}')
print('-' * 78)
print(tabla.T.to_string())
if tabla_ablacion is not None:
    print('-' * 78)
    print(tabla_ablacion.T.to_string())
if tabla_agente is not None:
    print('-' * 78)
    print(tabla_agente.T.to_string())
print('-' * 78)
print(latencia_recuperacion.to_string())
print('-' * 78)
print(tabla_ragas.to_string())
print('-' * 78)
print('Agente:', ' | '.join(f"{'necesita' if d['necesita'] else 'no necesita'} -> {d['veredicto']}"
                            for d in demos_agente))
print('=' * 78)

                                  RESUMEN M3                                  
Corpus            : 8 documentos, 23 chunks (size 500, overlap 80, top-k 4)
Eval set          : 15 casos | 10 gold, 3 herramienta, 2 adversarial
Generador         : Qwen/Qwen2.5-1.5B-Instruct
Juez v1.1         : Qwen/Qwen2.5-7B-Instruct (4bit) | prueba de cordura: superada
Juez M2 (original): Qwen/Qwen2.5-1.5B-Instruct | prueba de cordura: FALLIDA
------------------------------------------------------------------------------
Sistema                      Línea base M2 (sin RAG) RAG ingenuo (S07) RAG avanzado (híbrida+rerank) Sistema final (con herramienta)
D1 · similitud (sin válvula)                    0.41             0.615                         0.645                           0.651
D2 · juez v1.1 (1-5)                             2.0               4.4                           4.2                             4.2
D2 · juez M2 (1-5)                              1.73              3.53                       

## 12 · Lectura honesta

La versión completa está en `M3/README.md`. Resumen de las cuatro preguntas:

**Qué movió qué.** El salto grande es el del RAG frente a la línea base: de 2/10 a 8/10 casos gold y la válvula de 0/2 a 2/2. Del ingenuo al avanzado la mejora es de un caso (8 a 9 gold) y D3 se queda en 12/15. De las cinco hipótesis del diagnóstico, tres no se sostuvieron: para `-999`,
`PR-AUC` y la exclusión de `sexo` la búsqueda densa ya ponía primero el chunk correcto. La que sí se confirmó es Q07, y la ablación muestra que la arregla el reranker, no BM25: la fusión RRF deja el chunk en la posición 8, fuera del top-4, mientras que el reranker lo rescata con BM25 o sin él. Lo que aporta BM25 es que llegue el chunk exacto (doc6_c0) y de primero.

**Cuánto costó.** La recuperación sola pasa de 12,4 ms (densa) a 46,6 ms (híbrida + reranker), despreciable frente a una generación de 1,3 a 2 s. El sistema final sube la mediana a 1,77 s, y en los casos de cálculo llega a 4,6 s sin comprar ningún acierto.

**Qué falla queda.** Recuperación resuelta: el documento correcto llega en 13 de 13 casos y el
`context_recall` de RAGAS (0,923) lo confirma. Lo que queda es generación, con `faithfulness` en
0,79: la abstención indebida de Q02 y la aritmética de T01 y T02.

**La válvula.** Se activó en A01 y A02 en todas las versiones con RAG; la línea base falló las dos.
Pero Q02 muestra el reverso: una válvula que se dispara de más le niega al analista una respuesta que estaba en los documentos.

Verificable en `scorecard_m3_detalle.csv`, `caso_a_caso.csv`, `ablacion_m3.csv`,
`ablacion_agente_m3.csv` y `rangos_diagnostico.csv`.

---

*SI4006 · Universidad EAFIT · Entrega M3 — RAG agéntico.*

In [1]:
!zip -r m3_salida.zip m3_salida
from google.colab import files; files.download('m3_salida.zip')

'zip' is not recognized as an internal or external command,
operable program or batch file.


ModuleNotFoundError: No module named 'google.colab'